# Apache Flink

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-interview-questions/blob/master/notebooks/topics/flink.ipynb)

*How to use this notebook.* Read the explanation cells and run the code cells below them. Python examples list their imports, so install what a section needs before running it. SQL and shell snippets are illustrative: they show the syntax for the platform named in that section and need that platform (or a compatible engine) to run.

Apache Flink is a distributed, stateful stream processor with true event-time semantics, watermarks, keyed state, checkpointing, and exactly-once connectors. It runs the same operators over unbounded streams and bounded (batch) inputs. These questions cover the DataStream and Table APIs, event time, windowing, checkpointing and savepoints, state backends, exactly-once Kafka pipelines, and production trade-offs.

## When would you choose Flink over Spark Structured Streaming?

Choose Flink when you need true event-by-event stream processing, low latency, complex event-time handling, large stateful operations, or exactly-once streaming semantics with long-running jobs. Spark Structured Streaming is often simpler for micro-batch workloads and teams already standardized on Spark.

Example:

+ fraud detection with millisecond-level decisions: Flink
+ hourly incremental aggregation into a lakehouse: Spark Structured Streaming

The strongest answer compares latency, state management, operational complexity, connectors, and team skill set.

**The real decision is the execution model.** Flink is a pipelined, record-at-a-time streaming runtime. Each operator runs continuously, and state lives in the operator, protected by periodic checkpoints. Spark Structured Streaming runs micro-batches: every trigger it plans a batch, reads new offsets, computes, and commits. Latency therefore has a floor of one trigger interval, and state is carried between batches in a state store.

Pick Flink when per-event latency matters, when event-time correctness with late data matters, or when state is large and long lived. Pick Spark when you already run Spark, when you want one API for batch and stream, or when a one-minute micro-batch already meets the SLA.

Worked example: an ad-billing pipeline needs a p99 latency under 500 ms and must attribute clicks that arrive up to two minutes late. Flink with event time and 2-minute watermarks fits. An hourly revenue rollup does not need Flink; Spark or dbt is cheaper to operate.

**Common mistake:** assuming Flink is always faster. For batch-like, high-throughput ETL, Spark's batch engine often wins on cost per record because it amortizes scheduling and serialization across a whole batch.

## What are event time, processing time, and watermarks in Flink?

Event time is when the event actually happened. Processing time is when Flink processes the event. Watermarks tell Flink how far event time has advanced and how long to wait for late data before closing windows.

Example:

```text
event happened: 10:00:05
event processed: 10:02:10
watermark: events up to 10:01:00 are mostly complete
```

Watermarks are essential for correct windowed aggregations when events arrive late or out of order.

**Three clocks.** Event time is the timestamp inside the record, set by the producer. Ingestion time is when the record entered Flink. Processing time is the wall clock of the operator that handles it.

A watermark is a heuristic promise: no more events with a timestamp older than T should arrive. It is not enforced by the source. In event time, a window fires when the watermark passes the window end. The watermark at an operator is the minimum of its inputs, so one idle partition can hold everything back.

Worked example: records carry `event_ts`. Assign `WatermarkStrategy.for_bounded_out_of_orderness(Duration.of_seconds(5))`. A window `[10:00, 10:01)` fires when the watermark reaches 10:01, roughly five seconds after the newest event seen.

**Common mistake:** using processing time for anything you must replay or audit. Processing-time results depend on when the job ran. Event time is deterministic, but you pay for timestamp assignment and watermark tracking, and you must decide what happens to records that arrive after the window fired.

## See you using Flink, briefly introduce Flink?

Flink is a distributed, stateful stream-processing framework that treats batch as a bounded stream. A job is a dataflow of `source -> transformation -> sink`, and the same runtime handles unbounded (streaming) and bounded (batch) input.

The main APIs are the DataStream API (Java, Scala, Python), the Table API and SQL, plus domain libraries such as CEP and FlinkML. Flink provides event-time processing, watermarks, keyed state, checkpointing, and exactly-once semantics.

In [ ]:
from pyflink.datastream import StreamExecutionEnvironment

env = StreamExecutionEnvironment.get_execution_environment()
env.from_collection(["hello", "flink"]).print()
env.execute("flink-intro")

**Mental model of a job.** A Flink program is a dataflow graph: one or more sources, a chain of transformations, and one or more sinks. Operators can be parallel, and the ones after a `key_by` or `rebalance` are partitioned across the network. Nothing runs until `env.execute()` is called.

A saved program is compiled twice. The client builds a JobGraph, which is submitted to the JobManager. The JobManager builds the ExecutionGraph, which expands each operator into parallel subtasks and assigns them to slots. The physical execution is a set of tasks, each of which may contain several chained operators.

**Common mistake:** expecting side effects to happen while building the graph. Transformations are lazy; a `print` in a `map` runs only after `execute()`.

**How to read a job:** source -> map/filter (likely chained into the source task) -> key_by/window (new task, network shuffle) -> sink. Chained operators share a thread and a slot; a key_by breaks the chain.

## What are the differences between Flink and Spark Streaming?

1. The design ideas are different. Flink considers batch to be a kind of streaming, and spark considers a streaming batch.
2. The architecture model is different. Spark has Driver, Master, Worker, and Executor. Flink has the concepts of TaskManager, JobManager, Task, SubTask, and Slot
3. Flink's streaming data processing is much stronger than spark, for example, time supports three kinds of time
There are more windows than spark
4. In the case of out-of-order data, Flink is stronger than spark, because flink has watermark. In fact, the calculation method when running is the time of the last data-if watermaker is greater than the end of the window, execute
5. For fault tolerance, flink is also better than spark. For example, flink supports two-stage transactions to ensure that data after program crashes will not be re-consumed. Spark also has checkpoints, but it only ensures that data is not lost, and it cannot be repeated. consumption.

| Aspect | Spark Structured Streaming | Flink |
|---|---|---|
| Model | micro-batch | true record-at-a-time streaming |
| Latency | sub-second to seconds | milliseconds |
| Time semantics | event time with watermarks | event time, watermarks, three time notions |
| State | limited, epoch-based | large keyed state, RocksDB |
| Exactly-once | checkpoint + idempotent sink | checkpoint + two-phase-commit sinks |
| Batch | Spark SQL engine | bounded streams on the same runtime |

**Execution, not just API.** Flink injects checkpoint barriers into the live stream and snapshots operator state while data keeps flowing. Spark Structured Streaming commits each micro-batch: offsets, state updates, and output are tied to a batch id. Flink's latency is bounded by network and processing, typically milliseconds; Spark's is bounded by the trigger interval.

Flink's barrier alignment means an operator waits for all inputs to deliver the barrier. Under backpressure that wait grows, which is why unaligned checkpoints exist. Spark's batch boundary is a natural consistency point; it does not need barrier alignment.

**Common mistake:** comparing throughput at equal parallelism without accounting for Spark amortizing per-record costs over a batch. Measure both on your own data.

**When either works:** both give exactly-once with a replayable source and an idempotent or transactional sink. The difference is latency, state size limits, and operational cost.

## What are the Flink component stacks

1. The deployment mode can be Local, Cluster, Cloud
2. Then perform all basic services based on Runtime components
3. The upper layer provides DataStream API and DataSet API
4. At the high-level, there are Table API, CEP complex family care, ML machine learning, graph computing

**Four layers.** Deployment (standalone, YARN, Kubernetes, or a managed service) sits under the core runtime, which provides the distributed streaming dataflow with checkpointing and state. Above that sit the APIs: DataStream, Table/SQL, and the domain libraries. Connectors and formats attach to the API layer.

The DataStream API is the low-level, imperative layer. Table and SQL are declarative; a planner turns them into the same kind of dataflow. The old DataSet API was the batch API and is deprecated in favor of DataStream batch execution and SQL.

```yaml
# flink-conf.yaml: the runtime layer in practice
jobmanager.rpc.address: jobmanager
taskmanager.numberOfTaskSlots: 4
state.backend: rocksdb
execution.checkpointing.interval: 60s
```

**Common mistake:** assuming a SQL query maps one-to-one to operators. The planner may push down filters, combine aggregations, or introduce mini-batches, so the running topology can differ from what you wrote.

## Does Flink need to depend on Hadoop?

Flink can be completely independent of Hadoop and run without relying on Hadoop components, but as the basis of big data, it is conceivable that most big data frameworks cannot bypass it. Flink can support many components, such as Yarn's resource scheduling, HBase's real-time data storage and dimensional table access, HDFS state storage and offline computing, and so on.

**No.** Flink runs standalone or on Kubernetes without Hadoop. The Hadoop dependency only appears when you deploy on YARN, read or write HDFS, or use a connector that pulls in Hadoop classes.

If you do need it, keep the Hadoop jars off the Flink distribution and point Flink at an existing install with `HADOOP_CLASSPATH`, or use a shaded connector. This keeps the classpath small and avoids conflicting library versions.

**Common mistake:** dropping a random Hadoop version into `lib/`. The mismatch surfaces later as `NoSuchMethodError` or classloader conflicts, often only on the TaskManagers.

**Rule of thumb:** if you use S3 or Kafka and run on Kubernetes, you do not need Hadoop at all.

## How big is your FLink cluster?

This can be done according to the situation of your own company. Most companies use the yarn mode of deployment. Of course, yarn also has multiple modes of deployment, such as a single job or directly apply for a large amount of resources in yarn to be responsible. Job submission, most companies choose, should be single job mode.

**Size from three numbers, not one.** Required parallelism comes from peak throughput divided by per-core throughput, with headroom. TaskManagers and slots come from the required parallelism and the slots you can afford per JVM. Memory must cover the largest state you expect, because a RocksDB backend spills to local disk and slow disks stall checkpoints.

Worked example: peak 100k events/s, a single core sustains 20k events/s at your record size, so you need about 5 parallel subtasks. Add 2x headroom for spikes and recovery: parallelism 10. With 3 slots per TaskManager, that is 4 TaskManagers.

**Common mistake:** sizing only on throughput. A pipeline that processes 100k events/s but keeps 50 GB of keyed state needs disk and checkpoint bandwidth planning more than it needs more CPU.

```yaml
taskmanager.numberOfTaskSlots: 3
taskmanager.memory.process.size: 4096m
parallelism.default: 10
```

## What is the Flink programming model?

In fact, in one sentence,Source->Transformation->Sink*

**Stateful dataflow.** A program is a directed graph of sources, transformations, and sinks. Every transformation can be parallel, and every operator can hold state. Keyed state is partitioned by a key chosen with `key_by`; operator state belongs to the subtask and is not keyed.

The graph is built eagerly as objects but executed lazily. `env.execute()` triggers compilation, submission, and scheduling. That is why configuration such as checkpointing and the state backend must be set before `execute()`.

```python
from pyflink.datastream import StreamExecutionEnvironment

env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(4)
# ... build source, transformations, sink ...
# env.execute("orders")
```

**Common mistake:** calling `key_by` after a non-keyed window, or expecting a plain `map` to see per-key state. Only operators downstream of `key_by` get keyed state.

## What are the roles of Flink cluster? What are the functions?

Flink programs mainly have three roles: TaskManager, JobManager, and Client when they are running.
JobManager In the role of a manager in the cluster, it is the coordinator of the entire cluster. It is responsible for receiving the execution of Flink Job, coordinating checkpoints, recovering from failures, and managing Task Manager.
TaskManager It is responsible for the resource information on the node where the manager is located, such as memory, disk, and network. It will report the resource to the JobManager when it is started.
Client It is the client submitted by the Flink program. When a user submits a Flink program, a Client is first created. Then the program submitted by the user will be preprocessed and submitted to the cluster for processing.

**Control plane versus data plane.** The JobManager is the control plane: it schedules tasks, coordinates checkpoints, handles failures, and manages recovery. The dispatcher accepts submissions and starts a JobMaster per job. The ResourceManager talks to the cluster manager to acquire TaskManagers and slots. TaskManagers are the data plane and never route records through the JobManager.

Checkpoint coordination is a good way to see the split. The JobManager injects the barrier trigger into the sources and collects acknowledgements. Each TaskManager snapshots the state of its own tasks and writes it to durable storage.

**Common mistake:** scaling the JobManager for throughput. Throughput scales with TaskManagers and network; the JobManager only needs enough memory and CPU for scheduling and checkpoint metadata.

High availability (ZooKeeper or Kubernetes HA) replays the job from the last checkpoint when the JobManager fails.

## What is TaskSolt?

In Flink's architecture, TaskManager is the working node that is actually used to execute our program. TaskManager is a JVM process. In fact, in order to achieve the concept of resource isolation and parallel execution, the concept of TaskSlot was proposed at this time, which is actually In order to control how many Tasks the TaskManager can receive, the TaskManager is controlled by tasksolt, that is, if we have a source that specifies three parallelism, then he will use three solts, and the other one needs to be mainly parallel as an operator When the degree is the same, and there is no change in the degree of parallelism, or there is no shuffle, they will be together at this time. This is an optimized concept.

**A slot is a fixed slice of a TaskManager.** It has a share of managed memory, network memory, and one or more threads for running tasks. The number of slots per TaskManager is `taskmanager.numberOfTaskSlots`. Slots are the unit the scheduler allocates.

Slot sharing lets subtasks of different operators share one slot when they are in the same slot-sharing group and have the same parallelism. A slot therefore typically runs one parallel slice of the whole pipeline, not one operator.

```yaml
# four slots per TaskManager; the job can run up to four parallel slices per worker
taskmanager.numberOfTaskSlots: 4
```

**Common mistake:** treating slots as vCPUs. Slots are logical, so two CPU-heavy operators placed in the same slot still share the cores of that TaskManager. Size slots as a memory and isolation decision, and watch CPU contention in the Flink UI.

## What are the commonly used operators in Flink?

Map operator
Filter operator
KeyBy operator
Window window

In [ ]:
stream = env.from_source(source, watermark_strategy, "orders")

mapped = stream.map(lambda o: enrich(o))
flat = stream.flat_map(lambda o: o["items"])
filtered = stream.filter(lambda o: o["amount"] > 0)
keyed = filtered.key_by(lambda o: o["customer_id"])
counts = keyed.map(CountByKey())
joined = keyed.connect(other).key_by(lambda a: a["k"], lambda b: b["k"]).process(JoinFn())

Common shape: `source -> map/flatMap/filter -> keyBy -> window/process -> sink`. `keyBy` is what turns an operator into a stateful, parallel one, because state is scoped per key.

**Operator map.** Sources (`from_source`, Kafka, files), stateless transforms (`map`, `flat_map`, `filter`), partitioning (`key_by`, `rebalance`, `broadcast`, `forward`), stateful transforms (`reduce`, `aggregate`, `process`, `window`), and sinks (`sink_to`, Kafka, JDBC, files). `key_by` repartitions and always breaks operator chaining.

Chaining is why a simple pipeline is cheap. A source, a `map`, and a `filter` with the same parallelism and only forward edges are fused into one task, one thread, and one slot, so records never cross the network.

Worked example: `source -> map -> filter -> key_by -> window -> reduce -> sink` compiles into roughly two tasks: the fused source/map/filter, and the keyed window/reduce/sink chain.

**Common mistake:** putting a `rebalance` or `key_by` in front of every cheap operator and turning a single-threaded pipeline into a network-bound one. Only repartition when you need key locality or even distribution.

## What is the parallelism of Flink and What is the parallelism setting of Flink?

In fact, the parallelism of Flink is well understood. For example, kafkaSource, its parallelism is the number of partitions by default. The degree of parallelism is this operator, and how many tasksolt are needed, we should know that is the advantage of parallel computing. Generally, the degree of parallelism is set according to the amount of data. It is best to keep the source and map operators without shuffle, because the pressure on the source and map operators is not very large, but when our data table is widened, It is better to set it larger.

Parallelism is per operator and can be set globally, per job, or per operator. A Kafka source defaults to the topic's partition count.

In [ ]:
env = StreamExecutionEnvironment.get_execution_environment()
env.set_parallelism(8)                      # job-wide default
stream.key_by(lambda o: o["cust"]).map(Enrich()).set_parallelism(16)

# flink-conf.yaml
# parallelism.default: 8

```bash
# CLI override at submit time
flink run -p 16 -c com.acme.OrdersJob orders.jar
```

**Parallelism is per operator.** Each operator has a parallelism, which is the number of parallel subtasks. The effective value follows precedence: an operator-level setting wins, then the environment setting, then the client or CLI `-p`, then `parallelism.default`. The maximum parallelism ever used must be reflected in the slots and in `state.backend` configuration.

Sources are bounded by the input. A Kafka source with 12 partitions can use at most 12 useful source subtasks; any extra subtasks sit idle. Downstream keyed operators can use a higher parallelism because they repartition.

```yaml
parallelism.default: 16
taskmanager.numberOfTaskSlots: 4
# 4 TaskManagers provide 16 slots
```

**Common mistake:** setting `key_by` parallelism to 1, which funnels every key through one subtask and creates a permanent bottleneck. Also common: raising parallelism without raising Kafka partitions, so the extra subtasks do nothing.

## What is the relationship between Flink's Solt and parallelism?

Solt is a concept in TaskManager. Parallelism is a concept in the program, that is, the concept of execution level. In fact, Solt specifies how many slots this TaskManager has and how much parallelism can be supported, but the parallelism developed by the program uses slots That is Solt, that is, TaskManager is the provider and the program is the user

**Slots cap parallelism; parallelism does not cap slots.** The number of slots in the cluster is the ceiling on how many parallel subtasks can run at once. With slot sharing, one job's whole pipeline can occupy a slot, so a cluster with 16 slots can run a job with parallelism 16, provided no operator needs more.

Default parallelism should always be set explicitly. Do not rely on the runtime default; it varies with deployment mode and can silently leave cores idle or oversubscribe a small cluster.

Worked example: 3 TaskManagers x 4 slots = 12 slots. A job with `parallelism.default: 12` and no operator override fills the cluster with one parallel slice per slot. If one operator sets parallelism 16, the job cannot be scheduled and waits for resources.

**Common mistake:** assuming more slots always means more throughput. Slots share the TaskManager's CPU, disk, and network. Beyond the cores in the JVM, extra slots add contention, not capacity.

## What if Flink encounters an abnormal restart of the program?

Flink has some restart strategies, and as long as the checkpoint is done, it can be done at least once. Of course, it may not be accurate once, but some components can be done. The restart strategy generally set is a fixed delay restart strategy. The restart does not delete the checkpoint. Generally, the number of restarts set by our company is 4 times. If it stops, we will send a nail warning and start from the checkpoint when it starts.

Flink restarts failed tasks from the last completed checkpoint using a restart strategy. Set it per job:

In [ ]:
from pyflink.common import RestartStrategies

env.set_restart_strategy(
    RestartStrategies.fixed_delay_restart(4, 10_000)   # 4 attempts, 10 s delay (ms)
)
# or: RestartStrategies.failure_rate_restart(failure_rate, failure_interval, delay_interval)
# or: RestartStrategies.no_restart()

After the configured attempts are exhausted the job fails, which is when your alerting should fire and an operator restarts it from the latest savepoint.

**Restart is a state recovery, not a cold start.** On failure, the affected tasks are cancelled and the JobManager restarts them from the latest completed checkpoint. Sources rewind to the offsets stored in that checkpoint, operator state is restored from the snapshot, and processing resumes. The rest of the pipeline may keep running if only a region failed.

The restart strategy decides how often and how fast restarts happen. `fixed-delay` waits a fixed time between attempts. `failure-rate` restarts only if failures stay under a threshold. Exponential delay backs off under repeated failures.

```yaml
restart-strategy: fixed-delay
restart-strategy.fixed-delay.attempts: 3
restart-strategy.fixed-delay.delay: 10s
execution.checkpointing.tolerable-failed-checkpoints: 3
```

**Common mistake:** a permanent failure plus aggressive restarts creates a crash loop that floods the logs and never recovers. Use a bounded restart strategy and alert on repeated failures.

## Flink's distributed cache

The distributed cache implemented by Flink is similar to Hadoop. The purpose is to read the file locally and put it in the taskmanager node to prevent the task from repeatedly pulling data and reduce performance.

**One copy per TaskManager.** The distributed cache registers a file with the job. When a task starts, the TaskManager downloads the file once into its working directory and every subtask on that worker reads the local copy. This is how small static lookup files are shipped with a job.

In PyFlink you register the file with `env.register_cached_file(path, name)` and read it from the task's working directory. In Java you call `getRuntimeContext().getDistributedCache().getFile(name)`.

**Common mistake:** expecting the cache to refresh while the job runs. Files are fetched at task startup, so changing the file has no effect until the job restarts. For values that must change at runtime, use broadcast state or a lookup table, not the distributed cache.

## Broadcast variables in Flink

We know that Flink is parallel, and the calculation process may not be performed in a Slot. Then there is a situation: when we need to access the same data. Then the broadcast variable in Flink is to solve this situation. We can understand the broadcast variable as a public shared variable. We can broadcast a dataset, and then different tasks can be obtained on the node. There will only be one copy of this data on each node.

**Broadcast is a stream, not a variable.** You connect a small stream of rules or metadata to a main stream using `broadcast()` and `connect()`. A `BroadcastProcessFunction` reads the broadcast side and can emit enriched records from the main side. Each subtask stores its own copy of the broadcast state.

Broadcast state is operator state, not keyed state. It is held on heap, so it is meant for small, slowly changing data such as currency rates or feature flags. Broadcast state descriptors must match by name and type between the broadcast and the process function.

```python
# sketch: rules broadcast to every parallel subtask
# rules = env.from_source(...)
# main.connect(rules.broadcast(descriptor)).process(BroadcastEnrich())
```

**Common mistake:** broadcasting a large table. Every subtask holds a full copy, so memory grows with parallelism. For large dimension tables use a lookup join with a cached external table instead.

## Do you know what windows in Flink are?

Flink supports two ways to divide windows, according to time and count. session is also a kind of time
Tumbing Count Window： Perform calculation when reaching a certain number, no folding
Sliding Time Window： When a certain period of time is reached, roll over, there can be overlap, generally used to calculate the recent demand, such as nearly 5 minutes.
Tumbing time Window： When a certain period of time is reached, the slide is carried out, which can be thought of as the Nokia slide phone used before. This is actually a micro batch
Sliding Count Window： Slide when it reaches a certain number
Session Window: The window data has no fixed size, it is divided according to the parameters passed in by the user, and the window data does not overlap. It is similar to calculating the user's previous actions when the user logs out.

In [ ]:
from pyflink.common import Time
from pyflink.datastream import window

keyed = stream.key_by(lambda e: e["user_id"])
keyed.window(window.TumblingEventTimeWindows.of(Time.minutes(5))).aggregate(SumAmount())
keyed.window(window.SlidingEventTimeWindows.of(Time.minutes(15), Time.minutes(5))).process(TopN())
keyed.window(window.EventTimeSessionWindows.with_gap(Time.minutes(10))).process(Sessionize())
keyed.count_window(100).aggregate(SumAmount())   # count window

Time windows use event time with watermarks; count windows trigger on record count per key. Session windows close after a gap of inactivity and have no fixed size.

**A window is a way to group infinite streams by time or count.** Time windows come in tumbling (fixed, non-overlapping), sliding (fixed size, fixed slide), session (gaps of inactivity), and global (one window per key, closed by a trigger). Windows can be keyed or non-keyed; non-keyed windows use `window_all` and run at parallelism 1.

Each window creates state per key. That state is what makes windows expensive: many keys times many open windows. Allowed lateness keeps the window state alive after the window fires so late records can update results.

Worked example: `stream.key_by(lambda e: e["region"]).window(TumblingEventTimeWindows.of(Time.minutes(5)))` keeps one pane per region per five-minute interval, and emits when the watermark passes the end.

**Common mistake:** keying by a high-cardinality field such as a user id and using long sliding windows. The state can explode far beyond the input rate.

## Flink's state storage?

Flink often needs to store intermediate states during calculations to avoid data loss and recover from abnormal states. Choosing a different state storage strategy will affect the state interaction between JobManager and Subtask, that is, JobManager will interact with State to store state.
Flink provides three state storage:
MemoryStateBackend
FsSateBackend
RocksDBStateBackend

State backends decide where keyed state lives and how checkpoints are written. `HashMapStateBackend` keeps state on the JVM heap (fast, memory-bound), while `EmbeddedRocksDBStateBackend` spills to local disk and supports incremental checkpoints (large state). See the dedicated state backend question for configuration.

**Keyed state is scoped by key; operator state is scoped by subtask.** Keyed state descriptors (`ValueState`, `ListState`, `MapState`, `ReducingState`, `AggregatingState`) live in the backend and are partitioned by key. Operator state, including `ListState` used as union state and `BroadcastState`, is bound to the subtask and redistributed on rescale.

Which backend you choose changes the cost model. The heap backend keeps state in JVM objects and writes a full snapshot each checkpoint. RocksDB keeps state in a local embedded database and can write only changed files when incremental checkpoints are on.

**Common mistake:** treating state as free. A `MapState` with millions of keys on the heap causes long GC pauses and slow checkpoints. Move it to RocksDB, add state TTL, or redesign the key.

## What kind of time are there in Flink?

Event time: the time when the event actually occurred
Intake time: time to enter flink
Processing time: the time to enter the flink operator

```text
event time:      when the event actually happened (embedded in the record)
ingestion time:  when the event entered Flink
processing time: when a specific operator processes the event (wall clock)
```

Event time is the only notion that gives deterministic results, because it does not depend on when data arrives or how fast the job runs. Processing time is cheapest but non-reproducible.

**Event time is the default for correctness.** Event time uses timestamps carried in the record and is deterministic on replay. Ingestion time stamps records once when they enter the source. Processing time uses the operator's wall clock and is the cheapest but not reproducible.

You choose the clock at the window or timer level. A `TumblingEventTimeWindows` needs timestamps and watermarks; `TumblingProcessingTimeWindows` just needs the trigger to fire. Timers in `process` functions are registered against the same clock you select.

**Common mistake:** mixing clocks. Assign event-time watermarks at the source for one part of a pipeline and use processing-time windows in another, then compare results and wonder why they disagree. Pick event time for anything bounded by an SLA or replayed for audits.

## What is Watermark in Flink?

Watermark is an operation used by Flink to deal with out-of-order time. In fact, in Flink, if we use event time and take kafka's source, then the window execution time at this time is the smallest among the partitions Partitions are used for triggering, and each partition must be triggered to perform calculations. Why is this? In fact, it is because the partitions of Kafka are disordered. Orderly in the zone. The execution time is the maximum time minus the watermark>window end time, and the calculation will be executed at this time.

A watermark is a monotonically increasing timestamp that says "no event older than this should arrive". It drives window firing in event time.

In [ ]:
from pyflink.common import Duration, WatermarkStrategy
from pyflink.common.watermark_strategy import TimestampAssigner

class EventTimestamps(TimestampAssigner):
    def extract_timestamp(self, event, record_timestamp):
        return event["event_ts"]

wm = (
    WatermarkStrategy
    .for_bounded_out_of_orderness(Duration.of_seconds(5))
    .with_timestamp_assigner(EventTimestamps())
)
stream = env.from_source(source, WatermarkStrategy.no_watermarks(), "kafka-orders")
stream = stream.assign_timestamps_and_watermarks(wm)

With multiple input partitions, the operator's watermark is the minimum across inputs, so one idle partition can hold every window open; idle-source detection (`with_idleness`) fixes that.

**Watermarks flow like records with a timestamp.** A source's watermark strategy emits periodic watermarks. An operator with several inputs takes the minimum, because it can only advance as far as its slowest input. Watermarks never go backwards.

`for_bounded_out_of_orderness` trades latency for completeness: a larger delay means more late records are still accepted but windows fire later. `for_monotonous_timestamps` is the cheapest and assumes sorted input. `with_idleness` lets a partition that stops producing data be excluded so it cannot hold the watermark back.

```sql
CREATE TABLE orders (
  order_id BIGINT,
  region STRING,
  amount DECIMAL(10,2),
  order_ts TIMESTAMP(3),
  WATERMARK FOR order_ts AS order_ts - INTERVAL '5' SECOND
) WITH ('connector' = 'kafka', 'topic' = 'orders');
```

**Common mistake:** setting the watermark delay too small on noisy data. Records arriving after the window fires are dropped unless you configure allowed lateness or a side output.

## What is Unbounded streams in Apache Flink?

Any type of data is produced as a stream of events. Data can be processed as unbounded or bounded streams.
Unbounded streams have a beginning but no end. They do not end and continue to provide data as it is produced. Unbounded streams should be processed continuously, i.e., events should be handled as soon as they are consumed. Since the input is unbounded and will not be complete at any point in time, it is not possible to wait for all of the data to arrive.

**Unbounded means the input never ends.** A Kafka topic or a socket is unbounded; there is no terminal state to sort or materialize, so the job must run forever and recover from checkpoints. Unbounded jobs are normally run in streaming runtime mode.

Because the source never finishes, correctness depends on checkpointing. Without checkpoints, an operator failure means starting over from whatever offsets the source re-reads. With checkpoints, the source offsets are part of the snapshot, so recovery resumes near the failure point.

```yaml
execution.runtime-mode: STREAMING
execution.checkpointing.interval: 30s
```

**Common mistake:** treating an unbounded job like a batch job and never setting a checkpoint interval. The pipeline runs, but the first TaskManager loss restarts the whole thing from scratch, or with no checkpoint at all fails to recover.

## Apache Flink Job Execution Architecture

Processing unbounded data sometimes requires that events are consuming in a specific order, such as the order in which events arrives, to be able to reason about result completeness.

**Submission is a sequence of graphs.** The client turns your program into a JobGraph, where chains of operators are merged into single vertices. The JobManager builds the ExecutionGraph, which expands each vertex into parallel subtasks, and the scheduler assigns those subtasks to slots. Each assignment becomes a physical task.

Task lifecycle: a TaskManager receives a task, creates the operator instances, and connects its inputs and outputs to network channels. Deployment descriptors carry the job configuration and the state to restore. On failure the descriptor is reused, so recovery does not need a new submission.

**Common mistake:** reading the JobGraph as if it were the running topology. The ExecutionGraph is the one with parallelism, slots, and task states; a single JobGraph vertex may become many subtasks, or none while the job waits for slots.

## What is Bounded streams in Apache Flink?

Bounded streams have a beginning and an end point. Bounded streams could be processed by consuming all data before doing any computations. Ordered ingestion is not needed to process bounded streams since a bounded data set could always be sorted. Processing of bounded streams is also called as batch processing.

**Bounded input has an end, so the job can finish.** Files, a finite query result, or a Kafka topic read with an end offset are bounded. Flink can run bounded jobs in batch mode, where it knows the full input, sorts where useful, and can use blocking shuffles instead of pipelined ones.

Batch mode changes scheduling: stages can run one after another, and the scheduler may reuse resources. In streaming mode the same bounded job runs continuously but simply reaches the end and stops. Streaming mode is the default; choose batch mode when you want batch semantics and the input is finite.

```yaml
execution.runtime-mode: BATCH
```

**Common mistake:** running a large bounded backfill in streaming mode by default, then wondering why it holds far more state than a batch job would. Batch mode can release state after each stage boundary.

## What is Dataset API in Apache Flink?

The Apache Flink Dataset API was used to do batch operations on data over time. It was available in Java, Scala, and Python, and performed transformations such as filtering, mapping, aggregating, joining, and grouping. Note: the DataSet API is formally deprecated and was removed in Flink 2.0 — new jobs should use the DataStream API, Table API, or SQL instead.

```java
DataSet<Tuple2<String, Integer>> wordCounts = text
.flatMap(new LineSplitter())
.groupBy(0)
.sum(1);
```

**DataSet was the batch API.** It uses `ExecutionEnvironment`, `DataSet` transformations (`map`, `filter`, `join`, `groupBy`, `reduce`), and a lazy execution model like the DataStream API. Programs were compiled into an optimized batch plan with sorting and hash-based operators.

DataSet is deprecated (FLIP-131). Bounded DataStream programs and the Table/SQL API replace it. If you maintain old code, migrate to the DataStream API with `execution.runtime-mode: BATCH`, or to SQL.

**Common mistake:** trying to mix DataSet and DataStream in one job. They use different environments and cannot be connected directly; conversion goes through the Table API or a re-read of the data.

## What is DataStream API in Apache Flink?

The Apache Flink DataStream API is used to handle data in a continuous stream. On the stream data, you can perform operations such as filtering, routing, windowing, and aggregation. On this data stream, there are different sources such as message queues, files, and socket streams, and the resulting data can be written to different sinks such as command line terminals. This API is supported by the Java and Scala programming languages.

```java
DataStream<Tuple2<String, Integer>> dataStream = env
.socketTextStream("localhost", 9091)
.flatMap(new Splitter())
.keyBy(0)
.timeWindow(Time.seconds(7))
.sum(1);
```

In [ ]:
from pyflink.common import Types, WatermarkStrategy
from pyflink.datastream import StreamExecutionEnvironment

env = StreamExecutionEnvironment.get_execution_environment()
stream = env.from_source(source, WatermarkStrategy.no_watermarks(), "orders")

result = (
    stream.map(lambda o: (o["region"], o["amount"]), output_type=Types.TUPLE([Types.STRING(), Types.DOUBLE()]))
    .key_by(lambda t: t[0])
    .reduce(lambda a, b: (a[0], a[1] + b[1]))
)
result.print()
env.execute("regional-revenue")

The DataStream API gives explicit control over state, timers, watermarks, and connectors, which is why it is used for complex stateful pipelines where SQL is not expressive enough.

**The DataStream API is the typed, stateful core.** A `DataStream` is an immutable handle; each call returns a new stream and records an operator in the graph. `KeyedStream` comes from `key_by` and is the only place keyed state exists. `ConnectedStreams` and `BroadcastStream` are the other special streams.

The API exposes process functions for full control: `KeyedProcessFunction` gives you timers, keyed state, and side outputs. Simpler operators such as `map` and `filter` are shortcuts over the same runtime.

Worked example: a fraud rule needs to see the previous transaction per account. `key_by(account).process(MarkSuspicious())` keeps a `ValueState` per account, which a stateless `map` could never do because it does not know the key.

**Common mistake:** using `process` where a built-in `reduce` or `aggregate` would do. Aggregate functions keep bounded state; a `process` function that buffers every record grows without limit unless you add TTL or clear state.

## What is Apache Flink Table API?

Table API is a relational API with an expression language similar to SQL. This API is capable of batch and stream processing. It is compatible with the Java and Scala Dataset and Datastream APIs. Tables can be generated from internal Datasets and Datastreams as well as from external data sources. You can use this relational API to perform operations such as join, select, aggregate, and filter. The semantics of the query are the same if the input is batch or stream.

```scala
val tableEnvironment = TableEnvironment.getTableEnvironment(env)
// register a Table
tableEnvironment.registerTable("TestTable1", ...);
// create a new Table from a Table API query
val newTable2 = tableEnvironment.scan(TestTable1).select(...);
```

**One declarative query, two runtime modes.** The Table API and SQL describe a query over tables. In streaming mode the query is continuous and produces a changelog; in batch mode it runs once. The planner converts the query into the same kind of dataflow the DataStream API builds, with optimizations such as filter pushdown and projection pruning.

Streaming tables are dynamic. A query can produce insert-only rows or a changelog with updates and deletes. Sinks that can handle updates need a primary key; insert-only sinks such as files or Kafka append topics cannot consume retractions.

```sql
CREATE TABLE orders (
  order_id BIGINT,
  region STRING,
  amount DECIMAL(10,2),
  order_ts TIMESTAMP(3),
  WATERMARK FOR order_ts AS order_ts - INTERVAL '5' SECOND
) WITH ('connector' = 'kafka', 'topic' = 'orders',
        'properties.bootstrap.servers' = 'broker:9092',
        'format' = 'json');

SELECT region, SUM(amount) FROM orders GROUP BY region;
```

**Common mistake:** sending an updating aggregate to an append-only sink. The job fails or silently emits only some changes. Use an upsert sink or a windowed query that is insert-only.

## What is Apache Flink FlinkML?

FlinkML is the Flink Machine Learning (ML) library. It is a new initiative in the Flink community, with an expanding list of algorithms and contributors. FlinkML aims to include scalable ML algorithms, an easy-to-use API, and tools to help reduce glue code in end-to-end ML systems. Note: Flink Community has planned to delete/deprecate the legacy flink-libraries/flink-ml package in Flink1.9, and replace it with the new flink-ml interface proposed in FLIP39 and FLINK-12470.

**FlinkML is a legacy machine-learning library.** It offered a small set of algorithms and pipeline utilities built on the batch DataSet API, such as regression, clustering, and feature transformers. It targeted offline training over bounded data, not online learning inside a streaming job.

The library is effectively inactive, and it does not keep pace with current Flink releases. For machine learning today, train outside Flink (Spark MLlib, scikit-learn, a dedicated platform) and use Flink for feature computation or online inference.

**Common mistake:** choosing FlinkML for a new project because it sounds integrated. Check the release notes; if the library does not build against your Flink version, it is not a real option.

## What is Apache Flink?

The Apache Software Foundation created Apache Flink, an open-source, unified stream-processing and batch-processing framework. Apache Flink is built around a distributed streaming data-flow engine written in Java and Scala. Flink runs every dataflow programm in a data-parallel and pipelined fashion.

**Flink is a distributed engine for stateful computations over unbounded and bounded data.** Its defining traits are true streaming execution, event-time processing, exactly-once state consistency through checkpoints, and a layered API that ranges from DataStream to SQL.

The runtime is a dataflow engine. Sources and operators run as parallel tasks in TaskManagers, coordinated by a JobManager. State is held in the operators and snapshotted to durable storage, so a job can fail and recover without reprocessing everything.

**Common mistake:** describing Flink as "Spark but faster". The more useful distinction is that Flink treats streaming as the primary model and batch as a special case, while micro-batch engines treat streaming as a sequence of small batches.

## Explain Apache Flink Architecture?

Apache Flink is based on the Kappa architecture. The Kappa architecture uses a single processor - stream, who accepts all information as a stream, and the streaming engine processes data in real-time. Batch data in kappa architecture is a form of streaming data.
Apache Flink Architecture
The majority of big data frameworks are built on the Lambda architecture, which uses different processors for batch and streaming data. In Lambda architecture, batch and stream views have different codebases. The codebases must be combined in order to query and retrieve the results. Maintaining different codebases/views and combining them is a hassle, but the Kappa architecture fixes this problem by having only one real-time view, which eliminates the need for codebase merging.
The core principle of the Kappa architecture is to manage batch and real-time data via a single stream processing engine.
That is not to say that Kappa architecture is preferred to Lambda architecture; it is entirely dependent on the use-case and application to determine which architecture is optimal.

**Architecture in one pass.** A client submits a JobGraph to the Dispatcher, which starts a JobMaster. The JobMaster schedules the ExecutionGraph and asks the ResourceManager for slots. TaskManagers provide slots and run tasks; they exchange data directly over the network. Durable storage holds checkpoints and savepoints.

For high availability, JobManager state is stored externally (ZooKeeper or Kubernetes) so a standby can take over. TaskManagers are stateless between checkpoints: their task state is recovered from the checkpoint, not from the worker.

**Common mistake:** believing the JobManager participates in data transfer. It does not. If records are slow, look at the TaskManagers, the network, and the sinks, not the JobManager.

## Explain the Apache Flink Job Execution Architecture?

The Apache Flink job execution architecture is shown in the diagram below.Apache Flink Job Execution Architecture
Program
It is a piece of code that is executed on the Flink Cluster.
Client
It is in charge of taking code from the given programm and creating a job dataflow graph, which is then passed to JobManager. It also retrieves the Job data.
JobManager
It is responsible for generating the execution graph after obtaining the Job Dataflow Graph from the Client. It assigns the job to TaskManagers in the cluster and monitors its execution.
TaskManager
It is in charge of executing all of the tasks assigned to it by JobManager. Both TaskManagers execute the tasks in their respective slots in the specified parallelism. It is in charge of informing JobManager about the status of the tasks.

**From graph to running tasks.** The client fuses forward-connected operators into JobGraph vertices. The JobManager builds the ExecutionGraph, one ExecutionVertex per parallel subtask, connected by intermediate results. The scheduler deploys each subtask to a slot, and the task connects its inputs and outputs to network channels.

Exchanges are either pipelined or blocking. Streaming uses pipelined exchanges, where records flow as they are produced. Batch stages can use blocking exchanges, where the producer finishes before the consumer starts. This is why batch mode can spill and sort between stages.

**Common mistake:** assuming a failed subtask restarts only itself. With pipelined exchanges, a failure often restarts the whole pipeline region from the last checkpoint, because upstream buffers are part of the consistency boundary.

## What are the features of Apache Flink?

Flink uses common runtime for data streaming applications and batch processing applications.
It can run both Batch and Stream programs.
Provides APIs.
It has lightning fast speed.
Can process data in low latency.
It can easily integrate with big data tools like Apache Hadoop, Apache MapReduce, etc.
It is highly scalable.
It acts as a Fault tolerent.

**Feature list with mechanics.** True streaming (pipelined records), event time and watermarks, stateful operators with keyed and operator state, exactly-once checkpoints, savepoints for upgrades, backpressure-aware flow control, and layered APIs (DataStream, Table, SQL). It runs standalone, on YARN, or on Kubernetes.

Flow control is a real feature, not a slogan. A slow downstream operator stops reading, its input buffers fill, and upstream stops producing. That is backpressure, and Flink surfaces it in the UI instead of dropping records or growing unbounded queues.

**Common mistake:** assuming exactly-once covers everything. It covers the source, internal state, and sinks that support transactions or idempotent writes. A plain JDBC insert in the middle still duplicates on recovery.

## What is Storage and Streaming in Apache Flink?

Flink compose various capacity framework and can devour information from gushing frameworks.The following gushing framework, flink can persue compose information like:
FLume - Acts as an Aggregation Tool
HBase - Acts like NoSQL Databse in a Hadoop ecosystem
HDFS - Helps as Hadoop Distributed File System
Kafka - Helps in distributing messaging queue
RabbitMQ - Acts as a Messaging queue
S3 - It is Simple Storage Service from Amazon

**Flink stores state, not the stream.** Records stream through memory; only operator state is persisted, and only when a checkpoint completes. Durable storage such as S3 or HDFS holds the checkpoint and savepoint snapshots. RocksDB uses local TaskManager disk as a working area, but that disk is a cache of state, not a source of truth.

This is why a checkpoint must be durable before the job can recover. If the checkpoint storage is slow or unavailable, checkpoint duration grows, and eventually checkpoints time out. Source offsets live in the snapshot, so no separate offset store is needed.

**Common mistake:** treating local RocksDB files as a backup. If a TaskManager dies, its local state is gone; only the last completed checkpoint on remote storage can restore it.

## What is DataSet API?

DataSet API helps us in enabling the client to actualize activities like a guide, channel, gathering and so on.It is utilized for appropriated preparing, it is an uncommon instance of stream preparing where we have a limited information source.They are regular programs that implement transformation on data sets like filtering, mapping, etc. The DataSet API is deprecated (removed in Flink 2.0); batch work now runs on the DataStream API, Table API, or SQL.
Data sets are created from sources like reading files, local collections, etc.All the results are returned through sinks, the execution can happen in a local JVM or on clusters of many machines.

**DataSet was Flink's bounded-data API.** It reads a data source into a `DataSet` and applies transformations such as `map`, `filter`, `join`, and `groupBy`. Execution was lazy and compiled with a cost-based optimizer that chose join strategies and sort orders.

It is deprecated in favor of the DataStream API in batch mode and the Table API. The replacement keeps the same bounded semantics but uses one runtime and one set of connectors.

```java
// legacy DataSet style
ExecutionEnvironment env = ExecutionEnvironment.getExecutionEnvironment();
DataSet<String> lines = env.readTextFile("hdfs:///logs/");
lines.filter(s -> s.contains("ERROR")).print();
```

**Common mistake:** starting new work on DataSet. It is frozen; use DataStream batch mode or SQL instead.

## What are the DSL Tool's in Flink?

There are 3 types of DSL tools:
Table - Its is implanted in DataSet and DataStream API.As it helps in performing impromptu investigation utilizing SQL like articulation languague for preparing bunch and social strem.
Gelly - It enables clients to run a set of tasks and change the procedure of the diagram.
FlinkML - It is composed in Scala.It gives instinctive APIs and proficient calculation to deal with applications.

**Flink ships several query and processing DSLs.** Table API (programmatic relational), SQL (declarative, run through the SQL client or embedded), CEP (pattern matching over streams), and Gelly (graph algorithms, batch-oriented). The DataStream API is the imperative base they compile down to.

SQL is usually the fastest way to express windows, joins, and aggregations, while the DataStream API is where you go for custom state, timers, and unusual connectors. The two can be mixed through a TableEnvironment.

**Common mistake:** picking a DSL by familiarity instead of by need. A complex stateful event-time rule written as SQL often becomes unreadable; the same rule as a `KeyedProcessFunction` is explicit and testable.

## What are the Apache Flink domain-specific libraries?

The following is the list of Apache Flink domain-specific libraries.
FlinkML: It is used for machine learning.
Table: It is used to perform the relational operation.
Gelly: It is used to perform the Graph operation.
CEP: It is used for complex event processing.

**Libraries built on the core runtime.** CEP for pattern matching, Gelly for graph processing, FlinkML for legacy batch machine learning, and Stateful Functions (a separate project) for virtual-actor-style services. Connectors and formats form the other library surface.

CEP is the one most used in streaming jobs. It matches patterns such as "three failed logins within a minute" over an event-time stream and emits matches, handling out-of-order events with the watermark.

**Common mistake:** reaching for CEP for a rule a `KeyedProcessFunction` can express with one timer. CEP has its own state and matching semantics; simple thresholds are cheaper in plain state and timers.

## What is the programing model of Apache Flink?

The Apache Flink Datastream and the Dataset work as a programming model of flink and its other layers of architecture. The Datastream programming model is useful in real-time stream processing whereas the Dataset programming model is useful in batch processing.

```text
source (Kafka, files, collections)
   -> transformations (map, filter, keyBy, window, connect, process)
   -> sink (Kafka, filesystem, JDBC, Iceberg)
```

Each transformation becomes a chained operator where possible, and keyed operations carry state with them. Fault tolerance comes from checkpointing the state of the whole dataflow, not from replaying the source alone.

**Program equals graph plus state.** You build a dataflow from sources to sinks. Parallelism is per operator, state can be keyed or operator-scoped, and time can be event, ingestion, or processing. The graph is lazy until `execute()`.

The key mental shift from batch programming is that operators are long lived. A `map` that closes over a mutable dictionary is dangerous at parallelism greater than one, because each subtask has its own copy and there is no coordination. Use keyed state instead.

**Common mistake:** sharing mutable state across subtasks through a global variable. It works at parallelism 1 in a test and breaks silently when scaled out.

## What are the different types of tools supported by Apache Flink?

Apache Flink supports the below list of tools.
CLI(Command Line Interface
Apache Flink Dashboard
Apache Flink SQL Client
Scala Shell

**What ships with a distribution.** Connectors (Kafka, filesystem, JDBC, Elasticsearch, Kinesis), formats (JSON, CSV, Avro, Parquet, Debezium), a CLI for submission and savepoints, the SQL client, a REST API, and metric reporters (Prometheus, JMX, StatsD). The web UI is served by the JobManager.

Tools fall into three groups: build-time (API and connectors), submit-time (CLI, REST, SQL client), and run-time (UI, metrics, logs, savepoints). Most production incidents are diagnosed with the run-time group.

```bash
# CLI is the submit-time tool
flink run -c com.acme.OrdersJob -p 8 orders.jar
flink list
flink savepoint <jobId> s3://acme-flink/savepoints/
```

**Common mistake:** adding a connector jar to `lib/` for one job and then wondering why every job on the cluster now resolves a different version. Ship job dependencies as a shaded uber-jar instead.

## Can we consider Apache Flink as an alternative to Hadoop? if so then explain why?

Apache Flink provides more features and flexibility comparing to Hadoop because it uses the cycle dataflow programming model despite MapReduce which is a two-stage disk storage model, Flink programs can work in memory and can be used with other filesystems apart from Hadoop. Flink provides the unified framework using that we can create a single data flow that will use batch, streaming, machine learning, and SQL queries.

**They solve different problems.** Hadoop is storage (HDFS) plus a cluster resource manager (YARN) plus a batch processor (MapReduce). Flink is a processing engine. It can run on YARN and read HDFS, so it complements Hadoop instead of replacing it.

If the question means "can Flink replace MapReduce as the batch engine?", broadly yes. Flink handles bounded data with batch mode and often runs faster because it pipelines and avoids writing intermediate results to disk between every stage.

**Common mistake:** comparing Flink to the whole Hadoop stack. Comparing Flink to MapReduce as a processing engine is meaningful; comparing it to HDFS is not, because Flink needs durable storage for checkpoints and does not provide it.

## What are the use cases of Apache Flink?

Many real-world industries are using Apache Flink and their use cases are as mentioned below.

Financial Services
Financial industries are using Flink to perform fraud detection in real-time and send mobile notifications.

Healthcare
The hospitals are using Apache Flink to collect data from devices such as MRI, IVs for real-time issue detection and analysis.

Ad Tech
Ads companies are using Flink to find out the real-time customer preference.

Oil Gas
The Oil and Gas industries are using Flink for real-time monitoring of pumping and issue detection.

Telecommunications
The telecom companies are using Apache Flink to provide the best services to the users such as real-time view of billing and payment, the optimization of antenna per-user location, mobile offers, and so on.

**The pattern behind the use cases.** Flink pays off when the pipeline is stateful, event-time sensitive, and expected to run continuously: real-time analytics, continuous ETL and CDC, event-driven applications, anomaly and fraud detection, and feature computation for online models.

A useful test: if the answer changes depending on when late data arrives, or if you need per-key memory across events, Flink's model fits. If the answer is a periodic batch report, it does not.

**Common mistake:** using Flink as a message queue. Operators can buffer briefly, but Flink is not durable storage; if the job is down for a long time, the source must still hold the data to replay.

## What are the different ways to use Apache Flink?

Apache Flink can be deployed and configured in the below ways.
Flink can be setup Locally (Single Machine)
It can be deployed on a VM machine.
We can use Flink as a Docker image.
We can configure and deploy Flink as a standalone cluster.
Flink can be deployed and configured on Hadoop YARN and another resource managing framework.
Flink can be deployed and configured on a Cloud system.

**By API and by deployment mode.** APIs: DataStream (imperative), Table/SQL (declarative), and process functions for custom logic. Deployment modes: session (one long-running cluster shares resources across jobs), per-job (a cluster per job with its own JobManager), and application (the `main` runs on the cluster, which is the recommended mode for production).

Application mode avoids shipping the whole job jar from a client and gives each application its own JobManager, so resource accounting and classloading are clean. Session mode is convenient for interactive SQL but couples jobs on one cluster.

```bash
# application mode on Kubernetes
flink run-application -t kubernetes-application   -Dkubernetes.cluster-id=orders-job   -c com.acme.OrdersJob local:///opt/flink/orders.jar
```

**Common mistake:** running many production jobs on one session cluster with a shared JobManager. One heavy job's checkpoints and failures then affect the others.

## What is the different component stack of Apache Flink?

Apache Flink has a stack of components such as at storage level it supports different types of file such as HDFS, local file system, it also supports the database like HBase, MongoDB. Flink can be deployed in the standalone mode, cluster mode, and on the cloud system. It provides the dataset API for batch processing and the datastream API for stream processing also supports SQL, machine learning, and graph processing.

**Stack from bottom to top.** Deployment environment (standalone, YARN, Kubernetes), then the runtime (distributed dataflow, memory management, checkpointing), then the APIs (DataStream, Table, SQL), then the libraries (CEP, Gelly). Connectors sit beside the API layer and formats beside the connectors.

Each layer has its own failure mode. Deployment failures show up as unschedulable slots, runtime failures as checkpoint timeouts or backpressure, API failures as serialization or type errors, and library failures as wrong pattern matches.

**Common mistake:** debugging a serialization error by resizing the cluster. Match the symptom to the layer first; most type errors come from the API layer and are fixed in the job, not in the cluster.

## What is the command to start Apache Flink Cluster?

We can use the following command to start the Apache Flink Cluster.

```bash
cloudduggu@ubuntu:~/flink$ ./bin/start-cluster.sh
```

**Standalone mode uses the bundled scripts.** `bin/start-cluster.sh` starts one JobManager and one or more TaskManagers from `conf/masters` and `conf/workers`. `bin/stop-cluster.sh` stops them. On YARN or Kubernetes the cluster is created by the deployment command, not by these scripts.

```bash
$FLINK_HOME/bin/start-cluster.sh
$FLINK_HOME/bin/flink list
$FLINK_HOME/bin/stop-cluster.sh
```

The web UI is served at the JobManager's port 8081 by default. For a session cluster, submit with `flink run`; for application mode, use `flink run-application`.

**Common mistake:** starting only a JobManager and expecting jobs to schedule. Without TaskManagers registering, the job stays in `CREATED` or waits for slots. Check `flink list` and the TaskManager tab in the UI.

## What is the Apache Flink SQL client and how to start it?

Apache Flink SQL client provides a command-line interface to run SQL operations such as create a table, drop table, create a view, drop the view, and other SQL commands as well.
We can start the SQL client using the below command.

```bash
cloudduggu@ubuntu:~/flink/bin$ ./start-cluster.sh
cloudduggu@ubuntu:~/flink$ ./bin/sql-client.sh embedded
```

The SQL client runs DDL and queries against tables registered with a catalog, and it supports streaming queries with windowing.

```sql
CREATE TABLE orders (
  order_id     STRING,
  customer_id  STRING,
  amount       DECIMAL(12, 2),
  order_ts     TIMESTAMP(3),
  WATERMARK FOR order_ts AS order_ts - INTERVAL '5' SECOND
) WITH (
  'connector' = 'kafka',
  'topic' = 'orders',
  'properties.bootstrap.servers' = 'broker1:9092',
  'format' = 'json'
);

SELECT customer_id, SUM(amount) AS revenue
FROM orders
GROUP BY customer_id;
```

**The SQL client is an interactive SQL terminal.** Start it with `bin/sql-client.sh`. It can run embedded (starting a local mini-cluster) or connect to an existing cluster. In embedded mode the session is for exploration; production queries should be deployed as jobs against a real cluster.

You can run a script non-interactively with `bin/sql-client.sh -f job.sql`, which is how SQL jobs are usually deployed. Session settings such as checkpointing and state TTL are set with `SET`.

```bash
bin/sql-client.sh -f orders_job.sql
# or interactively
bin/sql-client.sh
```

```sql
SET 'execution.checkpointing.interval' = '30s';
SET 'table.exec.state.ttl' = '24h';
```

**Common mistake:** leaving a long query running in an embedded client and closing the terminal, which kills the job. Deploy SQL jobs with `-f` against a real cluster instead.

## What are the SQL statements supported in Apache Flink?

Apache Flink SQL supports the below list of statements.
SELECT query
CREATE TABLE, CREATE DATABASE, CREATE VIEW, CREATE FUNCTION
DROP TABLE, DROP DATABASE, DROP VIEW, DROP FUNCTION
ALTER TABLE, ALTER DATABASE, ALTER FUNCTION
INSERT TABLE
SQL HINTS
DESCRIBE
EXPLAIN
USE
SHOW

```sql
CREATE TABLE orders (...) WITH (...);
CREATE VIEW paid_orders AS SELECT * FROM orders WHERE amount > 0;
INSERT INTO revenue_sink SELECT customer_id, SUM(amount) FROM orders GROUP BY customer_id;
EXPLAIN SELECT * FROM orders;
SHOW TABLES;
```

**DDL, DML, and utility statements.** DDL creates tables, views, functions, and catalogs. DML is `INSERT INTO` plus `SELECT` with windowing, joins, and aggregations. Utilities include `SHOW`, `DESCRIBE`, `EXPLAIN`, and `SET`.

Flink SQL streaming DML is insert-oriented. You express updates as a changelog through a primary key and an upsert sink; you do not issue `UPDATE` or `DELETE` statements against a live stream.

```sql
EXPLAIN SELECT region, COUNT(*) FROM orders GROUP BY region;

CREATE VIEW v AS SELECT * FROM orders WHERE amount > 0;
```

**Common mistake:** running a bare `SELECT` and expecting a deployed job. In the client it prints results; to produce a pipeline you need `INSERT INTO sink SELECT ...` or `EXECUTE` for a submitted statement set.

## What is bounded and unbounded data in Apache Flink?

Apache Flink processes data in the form of bounded and unbounded streams. The bounded data will have a start point and an endpoint and the computation starts once all data has arrived. It is also called batch processing. The unbounded data will have a start point but no endpoint because it is streaming of data. The processing of unbounded data is continous and doesn't wait for complete data. As soon the data is generated the processing will start.

**Bounded data has an end; unbounded data does not.** The distinction affects scheduling and state. A bounded job can finish, release state at stage boundaries, and run in batch mode. An unbounded job runs until stopped and must checkpoint to recover.

The same program can often handle both. The source defines the nature of the data; `execution.runtime-mode` defines how Flink executes it. Batch mode on bounded input enables blocking shuffles and resource reuse.

**Common mistake:** assuming a Kafka source is always unbounded. It can be bounded by `set_bounded(OffsetsInitializer.latest())`, which makes it a finite read, often used for backfills and tests.

## What is the building block of Apache Flink streaming applications?

The applications which are executed on the Apache Flink cluster are defined by the following three building blocks for stream processing.
Streams
State
Time

**Streams, transformations, state, and time.** Everything else composes from these. A stream is a possibly infinite sequence of records. A transformation maps one or more streams to another. State is per-operator memory that survives one record to the next. Time and watermarks decide when event-time results are emitted.

Checkpointing ties the four together: it snapshots state and source positions so the whole composition can be replayed consistently. Windows are just a convenient combination of time, state, and triggers.

**Common mistake:** thinking only in transformations. The hard parts of a streaming job are state design and time semantics, because they determine both correctness and the cost of recovery.

## What are the most common types of applications that use the Apache Flink?

+ **Event-driven applications** such as fraud detection, anomaly detection, and rule-based alerting, where each event can trigger stateful logic.
+ **Data analytics applications** such as quality checks for telecom networks and product analysis for mobile apps.
+ **Data pipeline applications** such as continuously indexing an e-commerce catalog or feeding a lakehouse table.

**Four repeating shapes.** Continuous ETL and CDC ingestion into a warehouse or lake, real-time analytics and dashboards, event-driven applications with rules or patterns, and feature or metric pipelines serving online systems.

Each shape stresses a different part of Flink. ETL stresses source and sink throughput. Analytics stresses state size and query latency. Event-driven apps stress timers and CEP state. Feature pipelines stress freshness and exactly-once delivery to the online store.

**Common mistake:** treating a real-time dashboard and a billing pipeline as the same workload. The dashboard tolerates approximate, fast results; billing needs event-time completeness and replayability. They often need different watermark and late-data policies.

## What are the features of the Apache Flink execution Engine?

The core of the Apache Flink architecture is the robust, distributed, dataflow engine that performs execution as a stream using cycle dataflow execution. It provides the cost-based optimization for batch and stream programs for optimum processing and able to work in distributed memory.

**Execution-engine features.** Pipelined record transport, operator chaining to avoid unnecessary serialization, managed off-heap memory, backpressure-based flow control, asynchronous checkpointing, and task scheduling with slot sharing. Together these give low latency without unbounded buffering.

Chaining is the cheapest optimization. If a source, map, and filter share a slot and only forward records, they run in one thread and no record is serialized. Once you insert a `key_by`, the runtime must serialize and shuffle.

**Common mistake:** breaking chains accidentally with a no-op `rebalance` or `disable_chaining`, then paying serialization and network cost for data that never needed to move.

## What is Apache FlinkML?

Apache FlinkML is the machine learning library that provides APIs and scalable algorithms to perform the machine learning operations. FlinkML helps developers to reduce the Glue codes and test the code locally. The same code can be used by the developer to run on a cluster of the machine. Some of the algorithms of FlinkML are Supervised Learning, Unsupervised Learning, Data Preprocessing, and so on.

**A legacy library, not a core feature.** FlinkML provided batch machine-learning operators and a pipeline abstraction on top of the DataSet API. It was designed for offline training over bounded data and never became the main way to do ML on Flink.

For new work, train with a dedicated library and use Flink for the streaming side: computing features, joining them, and serving inference. If a project genuinely needs FlinkML, verify it builds against your Flink version first.

**Common mistake:** assuming "Flink + ML" means online learning inside the job. Most production setups train offline and only score in Flink, which keeps the model lifecycle out of the streaming job.

## How can I check the progress of an Apache Flink Program?

The best way to track the progress of the Flink program is the Apache Flink Dashboard web interface. It shows the jobs which are in running such as its state the resources allocated for that job and so on. We can start the Apache Flink Dashboard using "http://localhost:8081/".

Use the Web UI at `http://jobmanager:8081` for job state, per-operator records, backpressure, checkpoints, and the watermark per subtask. In production, scrape the REST API and metrics into Prometheus/Grafana.

```bash
curl -s http://jobmanager:8081/jobs/overview
curl -s http://jobmanager:8081/jobs/<jobId>/checkpoints | head
curl -s http://jobmanager:8081/jobs/<jobId>/metrics?get=numRecordsOut,currentInputWatermark
```

**Three views: UI, metrics, and CLI.** The web UI shows the job graph, per-operator records in and out, backpressure, checkpoint history, and watermarks. Metrics reporters push the same numbers to Prometheus or JMX. `flink list` and the REST API give job state from a script.

Backpressure is shown as a ratio per operator. A red or high backpressure value at an operator means its downstream is slow; the actual bottleneck is the first operator downstream that is busy but not backpressured.

```bash
flink list
curl -s localhost:8081/jobs/<jobId>/metrics?get=lastCheckpointDuration
```

**Common mistake:** reading "busy" as "slow". An operator that is busy and not backpressured is the bottleneck; one that is backpressured is just waiting on the operator after it.

## How Apache Flink handles the fault-tolerance?

Apache Flink manages the fault-tolerance of stream applications by capturing the snapshot of the streaming dataflow state so in case of failure those snapshots will be used for recovery. For batch processing, Flink uses the program’s sequence of transformations for recovery.

Flink injects checkpoint barriers into the stream. When a barrier reaches an operator, that operator snapshots its state; the JobManager collects all snapshots and marks the checkpoint complete. On failure, every task restarts from the last completed checkpoint.

In [ ]:
env.enable_checkpointing(60_000)   # checkpoint every 60s

```yaml
# flink-conf.yaml
execution.checkpointing.mode: EXACTLY_ONCE
execution.checkpointing.interval: 60s
execution.checkpointing.timeout: 10min
execution.checkpointing.min-pause: 30s
state.backend: rocksdb
state.checkpoints.dir: s3://acme-flink/checkpoints/
```

**Checkpoint and replay.** Sources inject barriers, operators snapshot state when the barrier passes, and the JobManager marks the checkpoint complete once all sinks acknowledge. On failure, tasks restore that state and the sources rewind to the stored offsets. Everything between two checkpoints is replayed.

Exactly-once holds when the source is replayable and the sink either supports two-phase commit or is idempotent. The state backend decides how the snapshot is written: heap state is copied in full; RocksDB can write only changed files.

```java
env.enableCheckpointing(60_000, CheckpointingMode.EXACTLY_ONCE);
```

**Common mistake:** assuming checkpointing alone gives exactly-once end to end. An external side effect such as an email or a non-transactional JDBC write will still duplicate on replay.

## How to run an Apache Flink program using CLI mode?

We can use Apache Flink CLI(command-line interface) tool to run the programs which are built as JAR files.
For example, we can run a "WordCounts.jar" JAR file using the below command.

```bash
cloudduggu@ubuntu:~/flink$ ./bin/flink run ./examples/batch/WordCounts.jar
```

**Submit, then track.** `flink run` uploads the jar, starts the job, and exits; the job keeps running on the cluster. Use `-c` for the entry class, `-p` for parallelism, and `-d` for detached mode so the CLI does not stream logs.

```bash
flink run -d -c com.acme.OrdersJob -p 8 orders.jar
flink list
flink cancel <jobId>
flink run-application -t kubernetes-application -c com.acme.OrdersJob local:///opt/flink/orders.jar
```

**Common mistake:** running without `-d` in a terminal and closing it. The client process may be part of the job lifecycle in some modes; detached submission avoids that coupling.

## What is the responsibility of JobManager in Apache Flink Cluster?

The Job Manager is responsible for managing and coordinating with distributed processing of a program. It assigns the task to node managers, handles the failures for recovery, and performs the checkpointing. It has three components namely ResourceManager, Dispatcher, and JobMaster.

**Coordination only.** The JobManager accepts the JobGraph, builds and schedules the ExecutionGraph, allocates slots through the ResourceManager, coordinates checkpoints, and drives recovery on failure. It also serves the UI and REST API.

It does not process records. It holds job metadata in memory, and for high availability writes it to a durable store so a standby can recover. Each job has its own JobMaster inside the JobManager process.

**Common mistake:** giving the JobManager a large heap "for state". Keyed state lives in TaskManagers; the JobManager only keeps scheduling and checkpoint metadata. Size it from the number of jobs and operators, not from state size.

## What is the responsibility of TaskManager in Apache Flink Cluster?

The Task Manager is responsible for executing the dataflow task and return the result to JobManager. It executes the task in the form of a slot hence the number of slots shows the number of process execution.

**Work and state.** A TaskManager is a JVM that owns a set of slots, provides managed memory, runs tasks, and stores operator state for those tasks. It exchanges data with other TaskManagers over the network and writes checkpoint data to durable storage.

TaskManager memory is split into framework heap, task heap, managed memory (used by RocksDB and batch operators), and network memory. `taskmanager.memory.process.size` sets the total; the fractions decide the split.

```yaml
taskmanager.memory.process.size: 4096m
taskmanager.memory.managed.fraction: 0.4
taskmanager.numberOfTaskSlots: 4
```

**Common mistake:** packing too many slots into one TaskManager. They share the JVM, its GC, its network buffers, and its local disk, so a heavy slot slows all the others.

## What is the filesystem supported by Apache Flink?

Apache Flink supports the following list of the filesystem.
local filesystem
Hadoop HDFS
Amazon S3
MapR FS
OpenStack Swift FS
Aliyun OSS
Azure Blob Storage

**Checkpoint and savepoint storage is pluggable.** Flink writes snapshots to a filesystem configured with `state.checkpoints.dir` and `state.savepoints.dir`: HDFS, S3, Google Cloud Storage, Azure Blob Storage, NFS, or the local filesystem. S3 and cloud stores need the matching filesystem plugin in `plugins/`.

Local filesystem storage is fine for a single-node test but not for production, because the state is not durable across workers. The storage must be shared by all TaskManagers and the JobManager.

```yaml
state.checkpoints.dir: s3://acme-flink/checkpoints/
state.savepoints.dir: s3://acme-flink/savepoints/
```

**Common mistake:** pointing checkpoint storage at a local path when running on Kubernetes. Pods are ephemeral and do not share disks, so recovery silently fails.

## What is the difference between stream processing and batch processing?

Batch processing consumes a bounded data set: all input is available before computation starts, so the engine can sort, optimize globally, and finish. Stream processing consumes an unbounded data set: results must be produced continuously as records arrive, with bounded state and event-time handling for out-of-order data.

Flink unifies both. A bounded job is simply a stream that ends, and the same operators run in both modes. The practical differences are latency, state size, and how the engine decides that a window is complete.

```text
batch:   read all input -> optimize -> compute -> write result
stream:  source -> operator -> operator -> sink   (runs forever, stateful)
```

**Latency, state, and completeness.** Stream processing handles unbounded data continuously with low latency; batch processing handles bounded data in one pass with high throughput and a clear end. The trade-off is when results are produced and how much state must be held.

Flink blurs the line: the same API runs both, and `execution.runtime-mode` selects batch or streaming semantics. Batch mode can sort, use blocking shuffles, and release state at stage boundaries. Streaming mode keeps pipelined exchanges and long-lived state.

**Common mistake:** using streaming mode for a bounded backfill and holding terabytes of state when batch mode would release it between stages. Match the runtime mode to whether the data ends.

## What can I do with my state?

Flink There are two main ways to use state ：
checkpoint Data recovery
Logical computing

**Read, update, expire, and snapshot it.** Keyed state is accessed through descriptors inside a `ProcessFunction` or `RichFunction`. You can read and write values, iterate map and list state, clear a key's state, and attach state TTL so old keys disappear.

Beyond the operator, state is visible through checkpoint and savepoint files. The State Processor API can read a savepoint into a dataset, modify it, and write a new savepoint, which is how you bootstrap state for a new job.

**Common mistake:** never clearing state. A per-user `ValueState` that is never removed grows forever, and checkpoints grow with it. Add TTL or clear state when the key is finished.

```python
# StateTtlConfig is attached to the descriptor's config, not the backend
# ttl = StateTtlConfig.new_builder(Time.days(7)).set_update_type(StateTtlConfig.UpdateType.OnCreateAndWrite).build()
```

## Flink watermark Transmission mechanism.

Flink Medium watermark Mechanism is used to deal with disorder ,flink It has to be event time , A simple example is , If the window is 5 second ,watermark yes 2 second , that All in all 7 second , When will calculation be triggered at this time , Suppose the initial time of the data is 1000, Then wait until 6999 It will trigger 5999 The calculation of windows , So the next one is 13999 Is triggered when 10999 The window of
In fact, this is watermark The mechanism of , In multi parallelism , For example, in kafka The window will not be triggered until all partitions are reached

**Watermarks travel with the data.** A source generates a watermark from its timestamp assigner, periodically, controlled by the auto-watermark interval. The watermark flows downstream along the same channels as records, and an operator with multiple inputs emits the minimum of the watermarks it has received.

Two consequences follow. First, a watermark never goes backwards, because older ones are ignored. Second, one silent input can hold back the whole job unless idleness is configured. The same minimum rule applies across Kafka partitions.

```yaml
pipeline.auto-watermark-interval: 200ms
```

**Common mistake:** expecting a per-partition window. Watermarks are combined by minimum, so a lagging partition delays every downstream window, not just its own records.

## Flink The time semantics of?

Event Time Time of occurrence
Ingestion time Events enter Flink Time for
processing time The time that the event enters the operator

**Three semantics, chosen per operator.** Event time reads the timestamp from the record, ingestion time assigns it once at the source, and processing time uses the operator clock. Event time is deterministic and replay-safe; processing time is cheapest and least reproducible.

Timers follow the same choice. A timer registered in event time fires when the watermark passes its timestamp; a processing-time timer fires after a wall-clock delay. Windows, joins, and CEP all inherit the time characteristic of their input.

**Common mistake:** using processing time for anything you must backfill or audit. Reprocessing the same data at a different time gives different results, which is usually not acceptable for billing or SLA reports.

## Flink window join:

1、window join, That is, according to the specified fields and scrolling sliding window and session window inner join
2、 yes coGoup In fact, that is left join and right join,
3、interval join That is to say In the window join There are some problems , Because some of the data really came after the meeting , It's still a long time , Then there will be interval join But it has to be the time of the event , And also specify watermark And water level and getting event timestamps . And set it up Offset interval , because join I can't wait all the time .

```java
// Tumbling window inner join: join on key and time window
left.join(right)
    .where(a -> a.key)
    .equalTo(b -> b.key)
    .window(TumblingEventTimeWindows.of(Time.minutes(5)))
    .apply(new JoinFunction<Event, Event, Joined>() {
        @Override
        public Joined join(Event a, Event b) { return new Joined(a, b); }
    });
```


These joins are part of the Java/Scala DataStream API; PyFlink does not expose `join`, `coGroup`, or `intervalJoin`.

Window joins require both sides to be keyed on the same expression and to share the window, which bounds the join state. Interval joins avoid buffering an entire window.

**Three ways to join streams.** Window joins pair records from two streams that fall in the same window and share a key; they are time-bounded but hold all records of the window. Interval joins pair each record with records from the other stream inside a time interval relative to its timestamp, which bounds state by the interval. Regular joins keep state indefinitely unless you add a TTL.

In Flink SQL, window joins are written as two windowed table functions on a common window and joined on `window_start`, `window_end`, and the key. Lookup joins consult an external table per record and are the usual choice for dimension data.

```sql
SELECT o.order_id, c.name
FROM TABLE(TUMBLE(TABLE orders, DESCRIPTOR(order_ts), INTERVAL '1' MINUTE)) o
JOIN TABLE(TUMBLE(TABLE clicks, DESCRIPTOR(click_ts), INTERVAL '1' MINUTE)) c
  ON o.order_id = c.order_id AND o.window_start = c.window_start;
```

**Common mistake:** a regular streaming join between two busy streams. State for both sides grows without bound; use a window or interval join, or set `table.exec.state.ttl`.

## Flink What are the window functions:

Tumbing window
Silding window
Session window
Count winodw

**Two families.** Grouped window aggregation (`GROUP BY window_start, window_end`) reduces each window to few rows. Window table-valued functions such as `TUMBLE`, `HOP`, and `CUMULATE` produce those window columns. Over aggregation (`ROW_NUMBER() OVER (...)`) computes rankings and running values within a partition.

Streaming over-aggregation needs an ordered, time-bounded input to be useful. `ROW_NUMBER` over a window is how TopN is written; without a window, the rank can change forever and the state never shrinks.

```sql
SELECT * FROM (
  SELECT order_id, region, amount,
         ROW_NUMBER() OVER (PARTITION BY region ORDER BY amount DESC) AS rn
  FROM orders
) WHERE rn <= 3;
```

**Common mistake:** running a global `ROW_NUMBER` TopN over an unbounded stream. The job keeps every row to maintain the ranking; add a window or an event-time bound, or the state grows without limit.

## keyedProcessFunction How it works

keyedProcessFunction There is one ontime Operation of the , If so event In time that The time to call is to look at ,event Of watermark Is it greater than trigger time Time for , If it is greater than, calculate it , No, just wait , If it is kafka Words , Then the default is to trigger the partition key in the shortest time .

**One record at a time, per key.** `KeyedProcessFunction` receives each element together with a context that exposes the current key, keyed state, timers, and side outputs. `process_element` returns nothing; output goes through `collect`.

Timers are registered against event or processing time. Each timer is identified by timestamp plus key, so registering the same timestamp for a key twice fires once. `on_timer` runs when the clock passes it, and it is where you flush buffered state or emit session results.

**Common mistake:** registering a new timer on every record without deduplicating. Thousands of duplicate timers slow the job; check state, or let the runtime coalesce timers at the same timestamp.

## How to deal with offline data such as the association with offline data?

1、async io
2、broadcast
3、async io + cache
4、open Method , Then the thread is refreshed at a fixed time , Cache updates are deleted first , Then write another one, and then write to the cache

Dimension enrichment from a database or object store is usually done one of four ways, in increasing order of freshness and cost:

+ **Async I/O**: query an external system per record without blocking the operator thread.
+ **Broadcast state**: broadcast a small dimension to every task.
+ **Async I/O plus cache**: add a TTL cache in front of the lookup.
+ **Scheduled refresh**: load the dimension in `open()` and refresh on a timer.

```java
public class DimLookup extends RichAsyncFunction<Order, Enriched> {
    private transient DatabaseClient client;
```


PyFlink does not expose `RichAsyncFunction`/`AsyncDataStream`; use the Java DataStream API above or a Table API lookup join.

**Match the join to the data size and freshness.** Small, slow-changing dimensions go into broadcast state or a lookup join with a cache. Large or versioned dimensions use a temporal join against a versioned table, where each fact is joined to the dimension row that was current at the fact's event time.

For high lookup volume, async I/O is the key optimization. A synchronous lookup blocks the task on every record; `AsyncDataStream` lets many lookups be in flight while other records keep processing.

**Common mistake:** a synchronous lookup join against a remote database under load. Latency per record times records per second quickly exceeds the task capacity, and backpressure spreads upstream. Use a cache or async I/O.

```sql
SELECT o.order_id, c.customer_name
FROM orders AS o
JOIN customers FOR SYSTEM_TIME AS OF o.order_ts AS c
  ON o.customer_id = c.customer_id;
```

## What if there is a data skew?

Flink How to view data skew ：
stay flink Of web ui You can see the data skew in , It's every one subtask The amount of data processed varies greatly , For example, some have only one M yes , we have 100M This is a serious data skew .
KafkaSource Data skew at the end
For example, upstream kafka It was specified when it was sent key There are data hotspots , So just after the access , Do a load balancing （ The premise is not keyby）.
Aggregation class operator data skew
Pre aggregation plus global aggregation

Look at the Flink Web UI: if one subtask processes far more records than its peers, that key group is hot. Two fixes:

+ **Pre-aggregate** locally before the global aggregation (a combiner), so the hot key sends fewer rows.
+ **Salt the key** for the hot keys, aggregate per salt, then aggregate again over the salts.

In [ ]:
# local pre-aggregation then global aggregation
partial = stream.key_by(lambda e: e["key"]).process(LocalAgg())
final = partial.key_by(lambda e: e["key"]).process(GlobalAgg())

# salted key for a small number of hot keys
import random
salted = stream.map(lambda e: (f"{e['key']}-{random.randint(0, 9)}", e))
agg = salted.key_by(lambda t: t[0]).reduce(sum_amount)
result = agg.map(lambda t: (t[0].rsplit("-", 1)[0], t[1])).key_by(lambda t: t[0]).reduce(sum_amount)

**Skew is uneven key distribution, and keyed state magnifies it.** One subtask owns the hot key, so it does the work while the rest idle. The symptom is one subtask with high backpressure or long checkpoint duration while others are fast.

Fixes, in order of preference: pre-aggregate locally before the keyed shuffle so fewer records reach the hot key; salt the hot key into several buckets, aggregate, then combine; or rebalance the stream if the work is not truly key-scoped. For a heavy dimension on one key, broadcast it instead of joining by key.

**Common mistake:** raising parallelism to fix skew. With a single hot key, the key still maps to one subtask, so extra parallelism does nothing. Fix the key or the aggregation, not the slot count.

## How to do dimension table Association?

1、async io
2、broadcast
3、async io + cache
4、open Method , Then the thread is refreshed at a fixed time , Cache updates are deleted first , Then write another one, and then write to the cache

For a slowly changing dimension, the two production patterns are a **lookup join** (Flink SQL, backed by an async lookup) and a **broadcast dimension** (Temporal Table Join).

```sql
-- Lookup join against a JDBC dimension with a cache
SELECT o.order_id, o.amount, c.region
FROM orders AS o
JOIN dim_customer FOR SYSTEM_TIME AS OF o.proc_time AS c
  ON o.customer_id = c.customer_id;
```

```sql
CREATE TABLE dim_customer (
  customer_id STRING, region STRING, updated_at TIMESTAMP(3),
  PRIMARY KEY (customer_id) NOT ENFORCED
) WITH (
  'connector' = 'jdbc',
  'url' = 'jdbc:postgresql://pg:5432/app',
  'table-name' = 'customer',
  'lookup.cache.max-rows' = '10000',
  'lookup.cache.ttl' = '10min'
);
```

**Four patterns.** Broadcast a small table to every subtask. Use a lookup join when the dimension lives in a database and you can tolerate per-record reads with a cache. Use a temporal join when dimension history matters. Use a bounded stream plus a regular join when the dimension is small and static for the job's life.

Freshness drives the choice. Broadcast state updates are pushed through a control stream and are immediate. Lookup joins see whatever the external table currently holds, which can be inconsistent with event time. Temporal joins are the only pattern that is correct for historical lookups.

**Common mistake:** using a lookup join for a versioned dimension. The current row is returned, not the row that was valid at event time, so historical facts get today's attributes and reporting quietly drifts.

## Flink checkpoint The overtime problem of How to solve?

1、 Is there a network problem
2、 Whether it is barrir problem
3、 see webui, Whether there is data skew
4、 If there's a data skew , So after solving the data skew , There will be improvements ,

Checkpoint timeouts almost always have one of four causes: backpressure, a slow external system inside the checkpoint, a barrier stuck behind a large in-flight batch, or RocksDB state that has grown too large.

```yaml
execution.checkpointing.timeout: 10min
execution.checkpointing.min-pause: 30s
execution.checkpointing.max-concurrent-checkpoints: 1
state.backend.incremental: true
state.backend.rocksdb.memory.managed: true
```

Diagnose with the Web UI: check the alignment time, the sync/async checkpoint durations, and whether one subtask is slow (data skew). Enabling unaligned checkpoints removes the barrier-alignment wait when backpressure is the cause.

**A checkpoint times out when it cannot finish inside `execution.checkpointing.timeout`.** The usual causes are large state, slow snapshot storage, barrier alignment stalled by backpressure, and long GC pauses. Start by reading the checkpoint history: duration, size, and alignment time tell you which cause applies.

Fixes map to causes. Incremental checkpoints and RocksDB reduce bytes written. Unaligned checkpoints remove the alignment wait under backpressure. A longer min-pause gives the job breathing room. Faster storage and smaller state reduce duration directly.

```yaml
execution.checkpointing.interval: 30s
execution.checkpointing.timeout: 10min
execution.checkpointing.min-pause: 10s
execution.checkpointing.unaligned.enabled: true
state.backend.incremental: true
```

**Common mistake:** raising the timeout as the only fix. If checkpoint size grows with state, longer timeouts just hide an unbounded growth problem until recovery itself becomes too slow.

## FlinkTopN And offline TopN The difference between?

topn It is a common function in both offline and real-time computing , It's different from... In offline computing topn, Real time data is continuous , This will give topn It's very difficult to calculate , Because it's going to keep a... In memory topn Data structure of , When new data comes , Update this data structure

**Streaming TopN is maintained, not recomputed.** Flink keeps an ordered state per partition and updates it as records arrive, emitting changes when the ranking shifts. Offline TopN recomputes the ranking from scratch on each run over a fixed dataset.

That difference shows up in cost and semantics. Flink holds the top N (or, worse, all candidates) per key in state. Offline jobs sort once and finish. Streaming results are emitted as an updating changelog, so the sink must handle updates.

**Common mistake:** using a global TopN without a time window. To know the top N of an infinite stream you must keep all candidates. Use a windowed TopN, or accept that state grows forever.

## Sparkstreaming and flink in checkpoint?

sparkstreaming Of checkpoint It will lead to repeated consumption of data
however flink Of checkpoint Sure Make sure it's accurate one time , At the same time, it can be incremental , fast checkpoint Of , There are three states ,memery、rocksdb、hdfs

**Different consistency mechanisms.** Spark Structured Streaming commits each micro-batch: offsets, state, and the write are tied to a batch id, and the checkpoint directory stores offsets and state versions. Flink injects barriers into the continuous stream and snapshots operator state plus source offsets for each checkpoint.

Recovery follows from that. Spark replays the batches after the last committed one. Flink replays the records between the last completed checkpoint and the failure. Spark's batch boundary gives a natural, slightly coarser consistency point; Flink's checkpoints can be taken more often without stopping the stream.

**Common mistake:** comparing checkpoint intervals directly. A Spark trigger controls both progress and consistency; a Flink checkpoint interval only controls the recovery point, and data keeps flowing during the snapshot.

## A brief introduction cep State programming:

Complex Event Processing（CEP）：
FLink Cep Is in FLink Complex time processing library implemented in ,CEP Allows event patterns to be detected in an endless stream of time , Give us a chance to grasp the important parts of the data , One or more time streams composed of simple events are matched by certain rules , Then output the data the user wants , That is, complex events that satisfy the rules .
Flink Data aggregation in , How to aggregate without windows
valueState Used to save a single value
ListState Used to hold list Elements
MapState Used to save a set of key value pairs
ReducingState Provided with ListState Same method , Return to one ReducingFunction The aggregated value .
AggregatingState and ReducingState similar , Return to one AggregatingState The value after internal aggregation

Flink CEP matches event patterns over a stream, and the underlying state is ordinary keyed state.

```java
Pattern<Event, ?> pattern = Pattern.<Event>begin("start")
    .where(evt -> evt.getAmount() > 10_000)
    .next("confirm")
    .where(evt -> evt.getAmount() > 10_000)
    .within(Time.minutes(5));
```


CEP lives in the Java/Scala `flink-cep` library; PyFlink has no `cep` module.

Flink exposes several state primitives: `ValueState` (one value), `ListState` (a list), `MapState` (a key-value map), `ReducingState` and `AggregatingState` (incrementally aggregated values).

**CEP keeps a non-deterministic finite automaton per key.** Patterns are declared with `begin`, `next`, `followed_by`, `within`, and quantifiers. Matching state lives in keyed state, and partial matches are held until they complete or time out.

Time bounds matter. `within(Time.minutes(5))` stops a partial match from waiting forever. Without a bound, an incomplete pattern occupies state indefinitely. Timed-out partial matches can be routed to a side output for analysis.

**Common mistake:** running CEP without a time bound on a high-cardinality key. State grows as incomplete patterns accumulate; bound the pattern and clean up timed-out matches.

## How to deal with abnormal data in Flink.

Abnormal data in our scenario , It is generally divided into missing fields and outlier data .
outliers ： For example, data on the age of the baby , For example, for the maternal and infant industry , The age of a baby is a crucial data , It's the most important , Because the baby is bigger than 3 At the age of 20, you hardly buy things from mothers and babies . There are days like ours 、 Unknown 、 And for a long time . This is an exception field , We will show the data to store managers and regional managers , Let them know how many ages are not allowed . If we have to deal with it , It can be corrected in real time according to the time of purchase , For example, maternity clothing 、 The rank of milk powder 、 The size of a diaper , As well as pacifiers, some can distinguish the age group to carry on the processing . We don't process the data in real time , We're going to have a low-level strategy task, night dimension, to run , Run once a week .
Missing field ： For example, some fields are really missing , If you can fix it, you can fix it . Give up if you can't fix it , It's like the news recommendation filter in the last company .

**Classify, then route.** Malformed records, schema mismatches, and nulls should not kill the job. Filter or map them to a side output that goes to a dead-letter sink, so the main pipeline keeps running and the bad records can be inspected and replayed.

Deserialization failures in a source are different from bad fields. Configure the connector's failure handling so a poison record does not restart the job forever. For late events, use allowed lateness or a side output rather than letting them silently disappear.

**Common mistake:** using `filter` to drop bad records with no trace. The job looks healthy while data quietly disappears; always send rejects somewhere durable.

## Is there any possibility of data loss in Flink?

Flink There are three kinds of data consumption semantics ：
At Most Once One consumption at most In case of failure, it may be lost
At Least Once At least once If there is a fault, it may be repeated
Exactly-Once Exactly once If something goes wrong , It can also ensure that the data will not be lost or repeated .
flink The new version is no longer available At-Most-Once semantics .

Flink interval join Can you write it simply

```java
KeyedStream<Event, String> keyed1 = ds1.keyBy(o -> o.getString("key"));
KeyedStream<Event, String> keyed2 = ds2.keyBy(o -> o.getString("key"));
// right.ts - 5s <= left.ts <= right.ts + 5s
keyed1.intervalJoin(keyed2)
    .between(Time.seconds(-5), Time.seconds(5))
    .process(new ProcessJoinFunction<Event, Event, Joined>() {
        @Override
        public void processElement(Event a, Event b, Context ctx, Collector<Joined> out) {
            out.collect(new Joined(a, b));
        }
    });
```

**Loss comes from three places, and checkpointing removes the first two.** 1) No checkpointing or a non-replayable source, so a failure cannot rewind. 2) A checkpoint that never completes, so there is no recovery point. 3) A sink that acknowledges before the data is durable, or a source that discards records before the checkpoint commits.

Exactly-once requires all three to align: replayable source, completed checkpoints, and a transactional or idempotent sink. At-least-once can duplicate but should not lose, provided the source can replay.

**Common mistake:** assuming exactly-once on internal state protects an external side effect. A non-transactional external write can be lost or duplicated even when the Flink state is consistent.

## At the time of submission How to make parallelism and how to allocate resources?

Parallelism based on kafka topic Parallelism of , A degree of parallelism 3 individual G

```bash
# per-job parallelism and TaskManager resources
flink run -p 16 \
  -D taskmanager.numberOfTaskSlots=4 \
  -D taskmanager.memory.process.size=4096m \
  -D jobmanager.memory.process.size=2048m \
  -c com.acme.OrdersJob orders.jar

# YARN: size the application cluster explicitly
flink run -m yarn-cluster -ynm orders-job -p 16 -ys 4 -yjm 2048m -ytm 4096m orders.jar
```

A Kafka source with `N` partitions gets at most `N` useful source parallelism, so align `-p` with the topic's partition count before scaling TaskManagers.

**Set parallelism per operator, then provision slots for the maximum.** Sources are bounded by partitions; keyed operators can be wider. The cluster must provide at least the largest operator parallelism in slots, or the job waits. Memory is provisioned per TaskManager and split between heap, managed, and network memory.

A practical sequence: size sources from Kafka partitions, size the busiest keyed operator from throughput, set `parallelism.default` to that value, then choose slots per TaskManager and the number of TaskManagers to cover it.

```bash
flink run -d -p 16 -c com.acme.OrdersJob orders.jar
```

```yaml
parallelism.default: 16
taskmanager.numberOfTaskSlots: 4
taskmanager.memory.process.size: 4096m
```

**Common mistake:** setting parallelism from CPU count alone. If Kafka has 8 partitions, 16 source subtasks leave half idle; only the downstream keyed stage benefits from the extra width.

## What's the common stream of API?

window join ah cogroup ah map flatmap,async io etc.

In [ ]:
env.from_source(...)                 # source connector
stream.map / flat_map / filter       # stateless transformations
stream.key_by(...)                   # keyed stream
stream.window(...) / process(...)    # windowed or stateful processing
stream.connect(other)                # two-stream operators, co-process
stream.union(other)                  # merge same-typed streams
# async I/O: Java AsyncDataStream; PyFlink has no async_io operator
stream.sink_to(...)                  # sink connector

**Stream types encode what you may do next.** `DataStream` supports stateless transforms and `key_by`. `KeyedStream` supports keyed state and windows. `WindowedStream` supports window functions. `AllWindowedStream` is the non-keyed variant and runs at parallelism 1. `ConnectedStreams` and `BroadcastStream` exist for two-input operators.

The type also controls chaining and partitioning. `key_by` returns a `KeyedStream` and forces a repartition; `connect` keeps two inputs with separate partitioning. `forward` and `rebalance` change distribution without changing the type.

**Common mistake:** calling `key_by` twice with different keys. The second key replaces the first and adds another repartition; keep one key per operator and derive what you need inside the function.

## How to maintain Checkpoint?

By default , If set Checkpoint Options ,Flink Only the most recently generated 1 individual Checkpoint. When Flink When the program fails , From the nearest one Checkpoint To recover . however , If we want to keep more than one Checkpoint, And can choose one of them to recover according to the actual needs , It's more flexible .Flink Support to keep multiple Checkpoint, Need to be in Flink Configuration file for conf/flink-conf.yaml in , Add the following configuration to specify that at most Checkpoint The number of .
For small files, please refer to The death of Daedalus - Solutions to the problem of small files in the field of big data .

Flink keeps only the latest completed checkpoint by default. Retain more if you want to choose a recovery point:

```yaml
# flink-conf.yaml
state.checkpoints.num-retained: 5
```

```bash
flink savepoint <jobId> s3://acme-flink/savepoints/
flink list
flink cancel -s s3://acme-flink/savepoints/ <jobId>
```

**Checkpoint health is a running concern.** Watch interval versus duration: if duration approaches the interval, checkpoints overlap and the job spends its time snapshotting. Watch size: steady growth means state that is never cleaned. Watch alignment time: growth there means backpressure is stalling barriers.

The maintenance levers are the interval, min-pause, timeout, concurrent checkpoints, incremental snapshots, and state TTL. Retention controls how many completed checkpoints are kept after a cancellation. For upgrades, use savepoints rather than depending on retained checkpoints.

```yaml
execution.checkpointing.interval: 30s
execution.checkpointing.min-pause: 10s
execution.checkpointing.max-concurrent-checkpoints: 1
execution.checkpointing.externalized-checkpoint-retention: RETAIN_ON_CANCELLATION
execution.checkpointing.tolerable-failed-checkpoints: 3
```

**Common mistake:** tuning the interval first. If state is the problem, a longer interval only spreads the pain; fix state size, add TTL, or move to RocksDB with incremental checkpoints.

## What's the difference at Spark and Flink Serialization?

Spark The default is Java Serialization mechanism , At the same time, there is an optimization mechanism , That is to say kryo
Flink It's a self implemented serialization mechanism , That is to say TypeInformation

Spark defaults to Java serialization (with Kryo as an option) and pays that cost per shuffled object. Flink uses `TypeInformation` and its own serializers, so it can serialize a record once, understand its schema, and compare keys without deserializing whole objects.

```java
// Flink needs type information to build serializers
DataStream<Tuple2<String, Integer>> s = env.fromElements(new Tuple2<>("a", 1));
```

**Flink leans on type information; Spark on generic serializers.** Flink extracts a `TypeInformation` for each record type from the function signatures. POJOs, tuples, primitives, and Avro or Protobuf types get efficient serializers. Anything it cannot analyze becomes a `GenericTypeInfo` handled by Kryo, which is slower and gives weaker guarantees.

Spark uses Java serialization or Kryo for most user types. Both can broadcast or pass closures, but Flink's closure cleaning is stricter because the job may run remotely.

**Common mistake:** using types Flink cannot analyze, such as nested private classes or collections of objects, and paying Kryo cost plus schema-evolution limits. Use POJOs with public fields or explicit Avro/Protobuf types when records cross the network.

In PyFlink the rows are serialized with cloudpickle, so the Java-side rules apply mainly to the JVM API and SQL connectors.

## How to deal with late data?

Flink Of watermark It's a delayed trigger mechanism .
commonly watermark Is and window To process out of order data ,Watermark The most fundamental is a time mechanism , For example, I set the maximum out of order time to 2s, The window time is 5 second , So when the event time is greater than 7s The window will be triggered when the window is opened . Of course, if there are data partitions , for example kafka Medium access watermake Words , that watermake It will flow , Take the smallest of all partitions watermake To flow , Because only the smallest can guarantee , All the previous data has come to , You can trigger the calculation .
use flink Can replace spark Batch processing function of
Flink The future goal is to integrate batch and stream processing , Because the batch data set you can understand is a limited data stream .Flink In terms of approval management , Especially this year Flink 1.9 Release after , Join in a large number of Hive Function of aspect , You can use Flink SQL To read Hive Metadata and datasets in , And use Flink SQL It is logically processed , But so far Flink Performance in batch processing , I can't do it Spark Of .
So far ,Flink There is still a lot to do in batch processing , Of course , If it's the introduction of real-time computing engine ,Flink Of course, it's the first choice .

Watermarks plus allowed lateness define the window's closing and grace period.

In [ ]:
stream.key_by(lambda e: e["user"]).window(
    TumblingEventTimeWindows.of(Time.minutes(5))
).allowed_lateness(120_000).side_output_late_data(late_tag).process(MyWindowFn())   # 2 minutes in ms

Records older than watermark minus allowed lateness go to the side output, which you can write to a quarantine sink and reconcile in a later batch job. Setting `allowed_lateness` to 0 gives the fastest results with the most late drops.

**Three levers, from cheapest to most correct.** Enlarge the watermark delay so fewer records are late. Allow lateness on the window so a late record still updates the result and re-emits it. Route records that exceed the allowed lateness to a side output for a separate reconciliation job.

The choice is a business decision. Billing may accept a long delay at the cost of latency. A live dashboard may prefer an early approximate result and no corrections. Record the policy where the pipeline is defined.

**Common mistake:** setting `allowed_lateness` without a sink that can handle updates. The corrected result is emitted, but an append-only sink turns it into a duplicate row instead of an update.

## When to use aggregate perhaps process:

aggregate： Incremental aggregation
process： Total polymerization
When calculating the accumulation operation, you can use aggregate operation .
When calculating the full amount of data in the window, use process, For example, sorting and other operations.

`aggregate` (or `reduce`) is an incremental aggregation: Flink keeps a small accumulator and updates it per record, so state stays constant. `process` with a window function buffers the window contents and lets you sort, iterate, and access timers, at the cost of state proportional to the window.

In [ ]:
# incremental: cheap in memory, no access to individual records
keyed.window(TumblingEventTimeWindows.of(Time.minutes(5))).aggregate(SumAmount())

# full window: needed for Top-N, sorting, or custom timers
keyed.window(TumblingEventTimeWindows.of(Time.minutes(5))).process(TopNPerRegion())

**Aggregate when the result is a function of the values; process when it depends on time and order.** `reduce`, `aggregate`, and `AggregatingState` keep one value per key, so state is bounded by the number of keys. `KeyedProcessFunction` exposes timers and lets you buffer records, which is necessary for pattern detection, sessionization, and custom emission.

The cost difference is large. An aggregate over a million keys holds a million small values. A process function that appends each record to a `ListState` holds every record until you clear it.

**Common mistake:** choosing `process` for convenience and then discovering the state cannot recover in reasonable time. If the logic is a running total, an aggregate is both simpler and cheaper.

## How does checkpointing work in Flink?

Flink injects checkpoint barriers into the stream. When a barrier reaches an operator, that operator snapshots its state; once every sink acknowledges its barrier, the JobManager marks the checkpoint complete. On failure, every task restarts from the last completed checkpoint.

In [ ]:
from pyflink.datastream import CheckpointingMode, ExternalizedCheckpointCleanup

env.enable_checkpointing(10_000, CheckpointingMode.EXACTLY_ONCE)
cfg = env.get_checkpoint_config()
cfg.set_min_pause_between_checkpoints(5_000)
cfg.set_checkpoint_timeout(60_000)
cfg.set_max_concurrent_checkpoints(1)
cfg.enable_externalized_checkpoints(ExternalizedCheckpointCleanup.RETAIN_ON_CANCELLATION)

Key knobs: interval (latency vs overhead), timeout (how long before a checkpoint is abandoned), min-pause (breathing room between checkpoints), and unaligned checkpoints (skip barrier alignment under backpressure).

**Alignment versus unaligned.** In an aligned checkpoint, an operator that receives a barrier on one input stops processing that input until every input has delivered its barrier, so the snapshot is a consistent cut. Under backpressure, that wait can dominate checkpoint duration.

Unaligned checkpoints let the barrier overtake buffered records. In-flight data is included in the snapshot, so the alignment wait disappears, at the cost of larger checkpoints. Exclude the alignment stall first, then decide whether the extra size is acceptable.

**Common mistake:** enabling unaligned checkpoints on a job whose real problem is slow state writes. Unaligned fixes barrier stalls, not storage throughput; checkpoint size grows instead.

## What are savepoints and how do you use them?

A savepoint is a manually triggered, externally owned snapshot used for upgrades, rescaling, and planned migration. Unlike a checkpoint, it is not deleted when the job stops, and it is the supported way to change job parallelism or topology.

```bash
# trigger a savepoint for a running job
flink savepoint <jobId> s3://acme-flink/savepoints/

# stop the job and take a savepoint first
flink stop --savepointPath s3://acme-flink/savepoints/ <jobId>

# resume from it, possibly with a different parallelism
flink run -s s3://acme-flink/savepoints/savepoint-xxxx -p 32 orders.jar
```

A savepoint only restores cleanly if the operator state is compatible; use `--allowNonRestoredState` only when you intentionally removed operators.

**Savepoints are a compatibility contract.** They are triggered manually, owned by you, and used for upgrades, rescaling, and migration. Restoring a savepoint checks that every operator in the new job maps to state in the savepoint. Renaming a UID without care breaks that mapping.

Always assign stable `uid` values to stateful operators before the first production run. With `uid`s fixed, you can change parallelism, swap a source, or reorder operators and still restore.

```bash
flink savepoint <jobId> s3://acme-flink/savepoints/
flink run -s s3://acme-flink/savepoints/savepoint-abc123 -p 32 orders.jar
```

**Common mistake:** relying on `--allowNonRestoredState` to skip mismatches. It hides unintended state loss; use it only when you are certain the dropped state is no longer needed.

## How do you configure the state backend?

State backends decide where keyed state lives and how snapshots are written.

```yaml
# flink-conf.yaml
state.backend: rocksdb
state.backend.incremental: true
state.checkpoints.dir: s3://acme-flink/checkpoints/
state.savepoints.dir: s3://acme-flink/savepoints/
state.backend.rocksdb.memory.managed: true
```

In [ ]:
from pyflink.datastream import StreamExecutionEnvironment, EmbeddedRocksDBStateBackend

env = StreamExecutionEnvironment.get_execution_environment()
env.set_state_backend(EmbeddedRocksDBStateBackend(enable_incremental_checkpointing=True))

`HashMapStateBackend` is fastest but bounded by heap; `EmbeddedRocksDBStateBackend` supports state larger than memory with incremental checkpoints, at the cost of local disk I/O. State TTL (`StateTtlConfig`) is what keeps long-running keyed state from growing forever.

**Configure at the cluster level and optionally per job.** `state.backend` selects heap (`hashmap`) or `rocksdb`. Incremental checkpoints require RocksDB and checkpoint storage that supports them, which cloud object stores and HDFS do. RocksDB memory is governed by managed memory so it does not fight the JVM heap.

After changing the backend, the next checkpoint must be a full one in some migration paths. Plan it as a savepoint-and-restart operation, not a hot switch.

**Common mistake:** enabling `state.backend.incremental` while writing checkpoints to a filesystem that does not support the required operations, then seeing every checkpoint fail. Verify the storage first.

## How do you build an exactly-once Kafka to Kafka pipeline?

End-to-end exactly-once needs three things: checkpointing enabled, a replayable source that stores offsets in Flink state, and a transactional sink that only commits when a checkpoint completes.

In [ ]:
from pyflink.common import WatermarkStrategy
from pyflink.common.serialization import SimpleStringSchema
from pyflink.datastream import StreamExecutionEnvironment, CheckpointingMode
from pyflink.datastream.connectors.kafka import (
    KafkaSource, KafkaSink, KafkaOffsetsInitializer,
    KafkaRecordSerializationSchema, DeliveryGuarantee,
)

env = StreamExecutionEnvironment.get_execution_environment()
env.enable_checkpointing(10_000, CheckpointingMode.EXACTLY_ONCE)

source = (
    KafkaSource.builder()
    .set_bootstrap_servers("broker1:9092")
    .set_topics("orders")
    .set_group_id("orders-flink")
    .set_starting_offsets(KafkaOffsetsInitializer.committed_offsets())
    .set_value_only_deserializer(SimpleStringSchema())
    .build()
)

sink = (
    KafkaSink.builder()
    .set_bootstrap_servers("broker2:9092")
    .set_record_serializer(
        KafkaRecordSerializationSchema.builder()
        .set_topic("orders-enriched")
        .set_value_serialization_schema(SimpleStringSchema())
        .build()
    )
    .set_delivery_guarantee(DeliveryGuarantee.EXACTLY_ONCE)
    .set_transactional_id_prefix("orders-flink-")
    .build()
)

stream = env.from_source(source, WatermarkStrategy.no_watermarks(), "kafka-orders")
stream.map(lambda v: v.upper()).sink_to(sink)
env.execute("orders-exactly-once")

Offsets are committed into the checkpoint (two-phase commit), and the Kafka transactions are committed only after the checkpoint succeeds, so a failure replays from the last checkpoint without duplicates downstream.

**The three ingredients, in order.** A Kafka source that stores offsets in Flink state and starts from committed or earliest offsets. Checkpointing enabled with `EXACTLY_ONCE` mode. A Kafka sink with `DeliveryGuarantee.EXACTLY_ONCE`, which uses Kafka transactions committed only when the checkpoint completes.

Because Kafka transactions are per checkpoint, the sink must be configured with a transaction timeout longer than the checkpoint interval plus recovery time, and consumers must read with `read_committed` isolation to avoid seeing uncommitted records.

**Common mistake:** a consumer that defaults to `read_uncommitted`. It can see records from a transaction that is later aborted, breaking the exactly-once illusion downstream.

## How do you do windowing in Flink SQL?

Flink SQL uses table-valued window functions, which are composable with normal `GROUP BY` and joins.

```sql
-- Tumbling window: one row per minute per region
SELECT window_start, window_end, region, SUM(amount) AS revenue
FROM TABLE(TUMBLE(TABLE orders, DESCRIPTOR(order_ts), INTERVAL '1' MINUTE))
GROUP BY window_start, window_end, region;

-- Sliding (HOP) window: emitted every minute over a 10-minute span
SELECT window_start, window_end, region, COUNT(*) AS orders
FROM TABLE(HOP(TABLE orders, DESCRIPTOR(order_ts), INTERVAL '1' MINUTE, INTERVAL '10' MINUTE))
GROUP BY window_start, window_end, region;

-- Cumulative window: running totals for the day
SELECT window_start, window_end, region, SUM(amount) AS revenue
FROM TABLE(CUMULATE(TABLE orders, DESCRIPTOR(order_ts), INTERVAL '1' HOUR, INTERVAL '1' DAY))
GROUP BY window_start, window_end, region;
```

Window results are emitted to the sink as the watermark advances past the window end; a record that arrives after its window has fired is dropped by default, and whether an already-emitted row can be corrected depends on the sink supporting retract/upsert updates.

**Window TVFs return extra columns.** `TUMBLE`, `HOP`, and `CUMULATE` wrap the table and expose `window_start` and `window_end`. Group by those columns plus your key. The result is emitted as the watermark advances past the window end.

Sinks decide the update semantics. A windowed aggregate is insert-only per window, so append sinks work. A non-windowed `GROUP BY` produces an updating changelog and needs an upsert-capable sink keyed by the group columns.

**Common mistake:** forgetting that a HOP window emits each record into multiple overlapping windows. The work and output multiply by the number of windows each record falls into.

## How do you handle event time and watermarks in PyFlink?

Assign timestamps and a watermark strategy at the source, so every downstream operator has a consistent notion of event time.

In [ ]:
from pyflink.common import Duration, WatermarkStrategy
from pyflink.common.watermark_strategy import TimestampAssigner
from pyflink.datastream import StreamExecutionEnvironment

env = StreamExecutionEnvironment.get_execution_environment()

class EventTimestamps(TimestampAssigner):
    def extract_timestamp(self, event, record_timestamp):
        return event["event_ts"]

watermarks = (
    WatermarkStrategy
    .for_bounded_out_of_orderness(Duration.of_seconds(5))
    .with_idleness(Duration.of_minutes(1))
)

stream = env.from_source(source, WatermarkStrategy.no_watermarks(), "kafka-orders")
stream = stream.assign_timestamps_and_watermarks(
    watermarks.with_timestamp_assigner(EventTimestamps())
)
keyed = stream.key_by(lambda e: e["user_id"])

`for_bounded_out_of_orderness(5s)` holds windows for five seconds past the newest event timestamp, and `with_idleness` prevents a silent partition from freezing all watermarks. Watermarks are per partition and the operator takes the minimum.

**Assign watermarks as early as possible.** Ideally the source carries a `WatermarkStrategy`, so timestamp assignment happens before any repartition and every downstream operator sees the same event time. Use `with_timestamp_assigner` when the timestamp is not a plain field.

For Kafka, prefer `KafkaRecordDeserializationSchema` that emits records with timestamps, or assign watermarks right after the source. Watermarks assigned after a shuffle are computed on the shuffled stream, which is usually still fine but harder to reason about.

**Common mistake:** assigning watermarks after a `key_by` or join. The source then has no event time, so any window or timer upstream of that point silently uses processing time or fails.

## What are Flink's trade-offs and when should you not use it?

+ **Performance:** millisecond latency and true event-time semantics, but throughput per core is lower than Spark's micro-batch and JVM/GC tuning matters.
+ **Resource contention:** state size, checkpoint duration, and TaskManager memory are tightly coupled; RocksDB state spills to local disk and a slow disk becomes a checkpoint bottleneck.
+ **Exactly-once:** guaranteed only when the source is replayable and the sink supports transactions or idempotent upserts. An external JDBC write is not transactional unless you make it idempotent.
+ **Operations:** savepoints, state schema evolution, and watermark/backpressure diagnosis require real operational maturity.

When NOT to use Flink: simple scheduled batch jobs (Spark/dbt are cheaper), pure ETL with no state or latency requirement, tiny data volumes, or teams without the operational capacity to run a stateful cluster.

**The operational bill.** Flink gives low latency and strong event-time semantics, but you pay with a stateful cluster, checkpoint storage, careful watermark policies, and a team that understands savepoints and backpressure. A small team running a nightly aggregation should not pay that bill.

A quick decision rule: if the pipeline needs per-key memory across events and a latency SLA under a minute, Flink is usually right. If it needs a periodic report over bounded data, a batch engine is cheaper and simpler.

**Common mistake:** adopting Flink for a workload that is really a sequence of batch jobs. The streaming job then runs mostly idle, holds state it does not need, and still requires checkpoint storage and on-call attention.

## Tuning: diagnosing backpressure and slow operators

**Start with the job graph, not the logs.** The Flink UI colors each operator by backpressure. A backpressured operator is waiting on something downstream; the bottleneck is the first operator downstream that is busy and not backpressured. Read the two metrics together: `busyTime` tells you the operator is doing work, `backPressuredTime` tells you it cannot write its output.

The ratio is what matters. `busyTime / (busyTime + backPressuredTime + idleTime)` near 1 with a low backpressure ratio means this operator is the constraint. A high backpressure ratio with low busy means look further downstream. Idle time near 1 means the operator is starved by its input.

**Worked example:** a window operator shows `backPressuredTime` at 85 percent, while the Kafka sink after it shows `busyTime` at 95 percent and near-zero backpressure. The sink is the bottleneck; the window is a victim. Adding parallelism to the window changes nothing.

**Checkpoint metrics separate two failure modes.** Slow checkpoints with small size point at storage or alignment. Slow checkpoints with large and growing size point at state that is never cleaned. The checkpoint history in the UI shows duration, size, and alignment time per completed checkpoint, which is enough to tell them apart.

When alignment time dominates, barriers are stuck behind buffered records, which is the signature of backpressure. When the synchronous phase dominates, an operator is doing too much work while holding up its own snapshot. When the asynchronous phase dominates, the snapshot storage is too slow.

**Common mistake:** treating every red operator in the UI as a problem to fix. Most red operators are just waiting. Fix the one at the end of the chain, then re-read the graph.

## Tuning: state backends and checkpointing

**Choose the backend from state size and churn.** `HashMapStateBackend` keeps state as JVM objects: fastest access, full snapshots, bounded by heap. `EmbeddedRocksDBStateBackend` keeps state in a local embedded database: larger-than-memory state and incremental checkpoints, at the cost of serialization and local disk I/O.

Incremental checkpoints write only files changed since the last checkpoint. They require RocksDB and durable storage that supports the operations, such as HDFS or S3. Heap state cannot be incremental; every checkpoint copies everything.

**Worked example:** 40 GB of keyed state on the heap means a 40 GB snapshot every checkpoint and a heap that must hold it. The same state on RocksDB with incremental checkpoints typically uploads only a few hundred MB per checkpoint, because most files are unchanged.

**Common mistake:** adding state TTL only after checkpoints become slow. Attach it when the state is designed, and choose an expiry that matches the business rule.

**Tune the checkpoint cadence as a budget.** Overhead is roughly checkpoint duration divided by checkpoint interval. A 30-second interval with a 10-second checkpoint spends a third of the time snapshotting. Min-pause keeps two checkpoints from running back to back. Concurrent checkpoints let a slow one overlap the next, which increases resource use but hides latency.

Unaligned checkpoints remove the alignment wait under backpressure and include in-flight data in the snapshot, so they grow the checkpoint. Enable them when alignment dominates; disable them when checkpoint size is the problem.

```yaml
state.backend: rocksdb
state.backend.incremental: true
state.checkpoints.dir: s3://acme-flink/checkpoints/
execution.checkpointing.interval: 30s
execution.checkpointing.timeout: 10min
execution.checkpointing.min-pause: 10s
execution.checkpointing.max-concurrent-checkpoints: 1
execution.checkpointing.unaligned.enabled: true
execution.checkpointing.tolerable-failed-checkpoints: 3
```

**Common mistake:** setting the interval shorter to get a better recovery point while checkpoints already take longer than the interval. Overlapping snapshots compete with processing and throughput falls.

## Tuning: parallelism, slots, and resources

**Set parallelism per stage from its real constraint.** Source parallelism cannot exceed Kafka partitions. Keyed stages scale with distinct keys, not with records. Stateless stages scale with cores. The cluster must provide slots for the largest stage, so an oversized stage holds the whole job back.

Slots per TaskManager are an isolation choice. One slot per core is a common start, but memory-heavy pipelines need more slots per JVM to amortize framework memory, and CPU-heavy pipelines want fewer to avoid contention. Managed memory is the share available to RocksDB and batch operators.

```yaml
parallelism.default: 16
taskmanager.numberOfTaskSlots: 4
taskmanager.memory.process.size: 8192m
taskmanager.memory.managed.fraction: 0.4
jobmanager.memory.process.size: 2048m
```

```bash
# command-line parallelism is the default for the submission
flink run -d -p 16 -c com.acme.OrdersJob orders.jar
```

**Resource sizing is a memory equation, not a core count.** `taskmanager.memory.process.size` must cover JVM overhead, framework heap, task heap, managed memory, and network buffers. If network memory is too small, tasks block on buffer acquisition and the job looks backpressured even when the CPU is idle.

A practical loop: measure the largest state per subtask, pick the backend, size managed memory for it, then size the heap for records and user objects, then leave headroom for network and framework memory. Re-check after a week of real traffic.

**Common mistake:** increasing slots to use idle cores without increasing memory. Each slot shares the same JVM heap and the same network buffer pool, so the extra subtasks make GC and buffering worse.

## Tuning: source and sink throughput

**Match source parallelism to partitions.** A Kafka source subtask reads one or more partitions; more source subtasks than partitions leaves readers idle. Check the partitions, not the topics, before setting source parallelism.

Watermarks need idleness to tolerate silent partitions, and the auto-watermark interval controls how often watermarks are emitted. A very small interval adds overhead; a very large one delays window firing.

```yaml
pipeline.auto-watermark-interval: 200ms
```

**Common mistake:** reading a topic with 6 partitions at source parallelism 24 and expecting 4x throughput. Only 6 subtasks do work; the rest wait and consume slots the downstream stages could use.

**Reduce per-record cost at the edges.** Lookups should be asynchronous so a task is not blocked per record. Buffers should be large enough to amortize the per-record cost but small enough to stay latency friendly; `execution.buffer-timeout` trades latency for throughput by waiting before flushing a partial buffer. Network memory sets how much in-flight data the cluster can hold.

Batching sinks turn many small writes into few large ones. JDBC sinks expose `sink.buffer-flush.max-rows` and `sink.buffer-flush.interval`; Kafka sinks have a transaction timeout that must exceed the checkpoint interval so transactions are not aborted early.

**Worked example:** 50k records/s at 500 bytes each is 25 MB/s. With a 100 ms buffer timeout, roughly 2.5 MB is in flight per output channel. If network memory per TaskManager is a few tens of MB, the default is usually sufficient; if you see buffer timeouts, raise network memory before raising parallelism.

**Common mistake:** increasing buffer timeout to raise throughput and then wondering why end-to-end latency grew by the same amount. Buffer timeout is a deliberate latency budget.

## Flink exercises

These exercises use only the Python standard library, so you can run them without a Flink cluster. Each one models a sizing or diagnosis decision you make in a real Flink job. Read the task, try it, then compare with the code and the expected result.

### Try it 1. Choose source parallelism from Kafka partitions

**Task.** Given a topic's partition count and a target parallelism from throughput, return the useful source parallelism. A Kafka source subtask can only read whole partitions, so parallelism cannot exceed the partition count.

**Hint.** The answer is the smaller of the partition count and the target parallelism. Also flag when the target is wasted because partitions are the binding constraint.

**Expected.** 24 partitions with target 16 gives 16 usable subtasks. 6 partitions with target 24 gives 6, and 18 subtasks would be idle.

In [ ]:
def source_parallelism(partitions, target):
    """Useful Kafka source parallelism: never more than the partitions."""
    return min(partitions, target)


def idle_subtasks(partitions, target):
    """Subtasks that would sit idle if parallelism is the target."""
    return max(0, target - partitions)


assert source_parallelism(24, 16) == 16
assert source_parallelism(6, 24) == 6
assert idle_subtasks(6, 24) == 18

print("24 partitions, target 16 ->", source_parallelism(24, 16))
print("6 partitions, target 24 ->", source_parallelism(6, 24),
      "(idle subtasks:", idle_subtasks(6, 24), ")")

### Try it 2. Compute checkpoint overhead from interval and size

**Task.** Given the checkpoint interval in milliseconds and a measured checkpoint duration, compute the fraction of wall-clock time the job spends snapshotting, including the min-pause.

**Hint.** Overhead is duration divided by the effective cycle, where the cycle is the interval plus min-pause when min-pause forces a gap. Keep the result as a fraction.

**Expected.** A 30-second interval with a 4.5-second checkpoint and 5 seconds of min-pause gives 4.5 / 35 = about 13 percent overhead.

In [ ]:
def checkpoint_overhead(interval_ms, duration_ms, min_pause_ms=0):
    """Fraction of time spent on checkpoints, including the min-pause."""
    cycle = interval_ms + min_pause_ms
    return duration_ms / cycle


overhead = checkpoint_overhead(30_000, 4_500, 5_000)
assert abs(overhead - 4_500 / 35_000) < 1e-9
assert overhead < 0.30

print(f"overhead = {overhead:.1%}")
print("within budget" if overhead < 0.20 else "consider a longer interval or smaller state")

### Try it 3. Detect backpressure from busy and idle metrics

**Task.** Given per-operator busy, backpressured, and idle times, compute the busy ratio and backpressure ratio, then name the bottleneck. The bottleneck is the operator that is busy and not itself backpressured.

**Hint.** An operator with backpressure is waiting downstream. The culprit is upstream of the backpressured ones, has a high busy ratio, and a low backpressure ratio.

**Expected.** In the sample, the window is heavily backpressured and the sink is busy with almost no backpressure, so the sink is the bottleneck.

In [ ]:
def ratios(busy, backpressured, idle):
    total = busy + backpressured + idle
    return busy / total, backpressured / total


operators = [
    ("source", 120, 20, 860),
    ("window", 150, 810, 40),
    ("sink", 920, 30, 50),
]


def find_bottleneck(ops):
    busy_ratio, bp_ratio = {}, {}
    for name, busy, bp, idle in ops:
        busy_ratio[name], bp_ratio[name] = ratios(busy, bp, idle)
    candidates = [n for n in busy_ratio if busy_ratio[n] > 0.8 and bp_ratio[n] < 0.1]
    return candidates[-1] if candidates else None


assert find_bottleneck(operators) == "sink"
for name, busy, bp, idle in operators:
    br, pr = ratios(busy, bp, idle)
    print(f"{name:8s} busy={br:.0%} backpressure={pr:.0%}")
print("bottleneck ->", find_bottleneck(operators))

### Try it 4. Size state from key count and per-key bytes

**Task.** Estimate heap state size from the number of keys and bytes per key, apply an overhead factor, and recommend a state backend. Use RocksDB when the estimate exceeds a heap budget.

**Hint.** Size = keys times bytes per key times an overhead factor (1.5 is a reasonable default for object headers and map entries). Compare it with the heap budget.

**Expected.** 2 million keys at 400 bytes with 1.5x overhead is 1.2 GB. With a 1 GB budget the recommendation is RocksDB.

In [ ]:
def state_size_bytes(keys, bytes_per_key, overhead=1.5):
    return int(keys * bytes_per_key * overhead)


def recommend_backend(keys, bytes_per_key, heap_budget_bytes):
    size = state_size_bytes(keys, bytes_per_key)
    return ("rocksdb" if size > heap_budget_bytes else "hashmap"), size


assert state_size_bytes(2_000_000, 400) == 1_200_000_000
backend, size = recommend_backend(2_000_000, 400, 1_000_000_000)
assert backend == "rocksdb"

print(f"estimated state = {size / 1e9:.2f} GB")
print("backend ->", backend)

### Try it 5. Set watermark delay from late-event distribution

**Task.** Given a list of observed lateness values in seconds, pick a watermark delay that covers the 99th percentile, rounded up to the next whole second. Report how many events it still misses.

**Hint.** Use a nearest-rank percentile: index = ceil(0.99 * n) - 1 on the sorted list. The delay is that value rounded up.

**Expected.** With lateness mostly under 4 seconds but a tail at 8 seconds, the delay is 8 seconds and covers at least 99 percent of events.

In [ ]:
import math


def percentile(sorted_values, p):
    k = max(0, math.ceil(p * len(sorted_values)) - 1)
    return sorted_values[k]


def watermark_delay(sorted_lateness_s, p=0.99):
    """Whole-second watermark delay covering the p-th percentile."""
    return math.ceil(percentile(sorted_lateness_s, p))


lateness = sorted([0, 1, 2, 1, 3, 2, 2, 1, 0, 4, 2, 3, 2, 1, 1, 2, 8, 2, 3, 2])
delay = watermark_delay(lateness)
covered = sum(1 for v in lateness if v <= delay) / len(lateness)

assert delay == 8
assert covered >= 0.99

print("watermark delay =", delay, "seconds")
print(f"covers {covered:.0%} of observed lateness")

### Try it 6. Plan a savepoint and upgrade

**Task.** Before an upgrade, compare the stateful operator UIDs in the running job with the new job. Restoring is safe when every old UID still exists. Report missing UIDs and whether the new job adds stateful operators with new UIDs.

**Hint.** Use sets. Missing old UIDs mean restore will fail unless you accept state loss with `--allowNonRestoredState`.

**Expected.** The sample plan keeps all old UIDs and adds one new one, so it restores cleanly without allowing non-restored state.

In [ ]:
def plan_upgrade(old_uids, new_uids):
    missing = sorted(set(old_uids) - set(new_uids))
    added = sorted(set(new_uids) - set(old_uids))
    safe = not missing
    return {
        "missing": missing,
        "added": added,
        "restore_ok": safe,
        "needs_allow_non_restored": not safe,
    }


old = ["source-orders", "window-revenue", "sink-warehouse"]
new = ["source-orders", "window-revenue", "sink-warehouse", "sink-alerts"]
plan = plan_upgrade(old, new)

assert plan["restore_ok"] is True
assert plan["added"] == ["sink-alerts"]
assert plan["missing"] == []

print("savepoint restore ok:", plan["restore_ok"])
print("new stateful operators:", plan["added"])

### Try it 7. Estimate network buffer pressure from records per second

**Task.** Estimate how much data is in flight on an output channel from the record rate, average record size, and buffer timeout. Compare it with the network memory available per TaskManager.

**Hint.** In-flight bytes = records per second times bytes per record times buffer timeout in seconds. If it exceeds the network budget, raise network memory or lower the timeout.

**Expected.** 50k records/s at 500 bytes with a 100 ms timeout is 2.5 MB in flight, comfortably under a 48 MB budget.

In [ ]:
def in_flight_bytes(records_per_s, bytes_per_record, buffer_timeout_ms):
    return records_per_s * bytes_per_record * (buffer_timeout_ms / 1000.0)


def pressure_ratio(records_per_s, bytes_per_record, buffer_timeout_ms, network_bytes):
    return in_flight_bytes(records_per_s, bytes_per_record, buffer_timeout_ms) / network_bytes


inflight = in_flight_bytes(50_000, 500, 100)
assert inflight == 2_500_000

ratio = pressure_ratio(50_000, 500, 100, 48 * 1024 * 1024)
assert ratio < 0.2

print(f"in-flight bytes = {inflight / 1e6:.1f} MB")
print(f"network pressure = {ratio:.1%} of budget")

## Real world use cases for Apache Flink

### 1. Real-time fraud detection

A payments company scores every card transaction within milliseconds and blocks suspicious ones. ~8,000 transactions/second (~700 million/day), 30-day rolling per-account state. Kafka carries transactions keyed by `account_id`; Flink keeps per-account velocity and amount state in RocksDB and writes alerts to Kafka and a case-management API.

In [ ]:
keyed = transactions.key_by(lambda t: t["account_id"])
alerts = keyed.process(VelocityDetector())   # timers + ValueState per account
alerts.sink_to(kafka_sink("fraud.alerts"))

Why: event time plus keyed state gives deterministic, replayable scoring, and exactly-once avoids double-blocking a customer after a failure.

### 2. Continuous lakehouse ingestion from CDC

A retailer streams Debezium change events into an Iceberg table with upserts. ~120,000 events/second, ~40 TB of current-state data, 5-minute checkpoints. Kafka source with committed offsets, Flink SQL `MERGE` (or the Iceberg sink), and hourly Iceberg compaction.

```sql
INSERT INTO lakehouse.customers
SELECT customer_id, name, email, updated_at
FROM cdc_customers;
```

In [ ]:
env.enable_checkpointing(300_000, CheckpointingMode.EXACTLY_ONCE)

Why: checkpointed Kafka offsets plus a transactional Iceberg sink make ingestion exactly-once, and Flink absorbs bursty CDC without a nightly rebuild.

### 3. Sessionization and product analytics

A media platform groups viewer events into sessions and computes engagement metrics. ~25,000 events/second, 10-minute session gaps, late-arriving mobile events up to 2 minutes old. Session windows with allowed lateness and a side output for late records.

In [ ]:
sessions = (
    events.key_by(lambda e: e["user_id"])
    .window(EventTimeSessionWindows.with_gap(Time.minutes(10)))
    .allowed_lateness(120_000)   # 2 minutes in ms
    .side_output_late_data(late_tag)
    .process(SessionMetrics())
)

Why: session windows match the business concept directly, and allowed lateness plus a side output turns "late" into a handled case instead of a silent drop.

### 4. Real-time alerting on infrastructure metrics

A SaaS provider detects anomalies in host and application metrics. ~500,000 samples/second, 1-second resolution, 15-minute sliding windows. Flink aggregates metrics, compares against a broadcast model, and emits alerts.

In [ ]:
windowed = (
    metrics.key_by(lambda m: m["host_id"])
    .window(SlidingEventTimeWindows.of(Time.minutes(15), Time.minutes(1)))
    .aggregate(MetricStats())
)
alerts = windowed.connect(model_stream.broadcast(state_descriptor)).process(AnomalyFn())

Why: a broadcast state stream lets the detection model update without redeploying the job, and sliding windows give continuous coverage instead of fixed buckets.

### 5. Exactly-once enrichment pipeline to Kafka

A logistics company enriches shipment events with customer and route data and republishes them. ~15,000 events/second, JDBC dimensions with a 10-minute cache, exactly-once output. Async lookup join for the dimension, Kafka source and transactional Kafka sink.

In [ ]:
env.enable_checkpointing(10_000, CheckpointingMode.EXACTLY_ONCE)
enriched = orders.key_by(lambda o: o["customer_id"]).map(AsyncDimLookup())
enriched.sink_to(exactly_once_kafka_sink("shipments-enriched"))

Why: async I/O keeps the external lookup off the operator thread, and transactional Kafka writes plus checkpointed offsets give true end-to-end exactly-once.